# COLAB - QISKIT - QAOA
## Luca Gandolfi
### QAOA Assignment

In [1]:
"""
Entry point del progetto di Quantum Optimization per l'allocazione VM-Server.

Esegue l'intero flusso:
    1. Costruzione del QuadraticProgram (con DOcplex e fallback manuale)
    2. Analisi della conversione QP → QUBO → Ising
    3. Risoluzione con ADMM classico (NumPyMinimumEigensolver)
    4. Risoluzione con ADMM + QAOA (StatevectorSampler, reps=2)
    5. Ispezione dei sottoproblemi ADMM (QUBO e convesso)
    6. Visualizzazione risultati, convergenza e confronto
"""

"\nEntry point del progetto di Quantum Optimization per l'allocazione VM-Server.\n\nEsegue l'intero flusso:\n    1. Costruzione del QuadraticProgram (con DOcplex e fallback manuale)\n    2. Analisi della conversione QP → QUBO → Ising\n    3. Risoluzione con ADMM classico (NumPyMinimumEigensolver)\n    4. Risoluzione con ADMM + QAOA (StatevectorSampler, reps=2)\n    5. Ispezione dei sottoproblemi ADMM (QUBO e convesso)\n    6. Visualizzazione risultati, convergenza e confronto\n"

In [2]:
from __future__ import annotations

import sys
import os

# Aggiungi la directory corrente al path
#sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

In [3]:
#from problem_formulation import (
#    build_quadratic_program_docplex,
#    analyze_qubo_conversion,
#    DEFAULT_M, DEFAULT_N, DEFAULT_PI, DEFAULT_PD, DEFAULT_C, DEFAULT_U,
#)
#from admm_solver import solve_classical, solve_qaoa
#from inspect_subproblems import inspect_admm_result, print_admm_decomposition_summary
#from results_analysis import (
#    decode_solution,
#    print_allocation_table,
#    compare_results,
#    plot_convergence,
#)

# ──────────────────────────────────────────────────────────────────────────────
# UTILITY
# ──────────────────────────────────────────────────────────────────────────────

In [4]:
def _to_serializable(obj):
    """Converte numpy types in tipi Python nativi per JSON."""
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.bool_):
        return bool(obj)
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

In [5]:
def _extract_admm_data(result, decoded):
    """Estrae dati strutturati da un ADMMResult per il salvataggio JSON."""
    s = decoded["s"]
    v = decoded["v"]
    l_vals = decoded.get("l", [])

    residui, n_iter, converged = [], 0, False
    state = getattr(result, "state", None)
    if state is not None:
        if hasattr(state, "residuals") and len(state.residuals) > 0:
            residui = [float(r) for r in state.residuals]
            n_iter = len(residui)
        if hasattr(state, "converge"):
            converged = bool(state.converge)

    cpu_usage = [
        float(sum(DEFAULT_U[j] * v[j][i] for j in range(DEFAULT_N))) if s[i] == 1 else 0.0
        for i in range(DEFAULT_M)
    ]
    e_idle = sum(DEFAULT_PI[i] * int(s[i]) for i in range(DEFAULT_M))
    e_dyn  = sum(
        DEFAULT_PD[i] * DEFAULT_U[j] * int(v[j][i])
        for i in range(DEFAULT_M) for j in range(DEFAULT_N)
    )

    return {
        "valore_obiettivo":      float(result.fval),
        "iterazioni":            n_iter,
        "converged":             converged,
        "residui_primali":       residui,
        "s_servers":             [int(x) for x in s],
        "v_allocation":          [[int(x) for x in row] for row in v],
        "l_continuous":          [float(x) for x in l_vals],
        "cpu_usage_per_server":  cpu_usage,
        "energia_idle":          float(e_idle),
        "energia_dynamic":       float(e_dyn),
        "energia_totale":        float(e_idle + e_dyn),
    }

# ──────────────────────────────────────────────────────────────────────────────
# FASI
# ──────────────────────────────────────────────────────────────────────────────

In [6]:
def print_banner():
    print("╔" + "═" * 68 + "╗")
    print("║   QUANTUM OPTIMIZATION — ALLOCAZIONE VM A SERVER FISICI          ║")
    print("║   ADMM con QAOA per minimizzazione consumo energetico            ║")
    print("╚" + "═" * 68 + "╝")

In [7]:
def fase1_build_problem():
    """Costruisce il QuadraticProgram e lo stampa."""
    print("\n" + "▶" * 3 + " FASE 1: Costruzione del QuadraticProgram")

    qp = build_quadratic_program_docplex()

    print("\n  Problema originale:")
    print(f"    M = {DEFAULT_M} server, N = {DEFAULT_N} VM")
    print(f"    P^I (idle)     = {DEFAULT_PI}")
    print(f"    P^D (dinamico) = {DEFAULT_PD}")
    print(f"    C (capacità)   = {DEFAULT_C}")
    print(f"    u (carico VM)  = {DEFAULT_U}")
    print("\n  Formulazione QuadraticProgram:")
    print(qp.prettyprint())

    return qp

In [8]:
def fase2_qubo_conversion(qp):
    """Converte QP → QUBO → Ising e restituisce qubo_info."""
    print("\n" + "▶" * 3 + " FASE 2: Conversione QP → QUBO → Ising")
    return analyze_qubo_conversion(qp)

In [9]:
def fase3_classical_solver(qp):
    """Risolve il problema con ADMM classico."""
    print("\n" + "▶" * 3 + " FASE 3: Risoluzione ADMM — Solver Classico")
    return solve_classical(qp)

In [10]:
def fase4_qaoa_solver(qp):
    """Risolve il problema con ADMM + QAOA."""
    print("\n" + "▶" * 3 + " FASE 4: Risoluzione ADMM — QAOA (quantistico)")
    return solve_qaoa(qp)

In [11]:
def fase5_inspect(result_classical, result_qaoa, qp):
    """Ispeziona i sottoproblemi ADMM di entrambe le soluzioni."""
    print("\n" + "▶" * 3 + " FASE 5: Ispezione sottoproblemi ADMM")

    info_classical = inspect_admm_result(result_classical, qp, label="ADMM Classico")
    info_qaoa      = inspect_admm_result(result_qaoa,      qp, label="ADMM + QAOA")
    print_admm_decomposition_summary(info_classical)

    return info_classical, info_qaoa

In [12]:
def fase6_analyze(result_classical, result_qaoa):
    """Decodifica, stampa e confronta i risultati; salva il plot di convergenza."""
    print("\n" + "▶" * 3 + " FASE 6: Analisi risultati e visualizzazione")

    print("\n  ── Allocazione Classica ──")
    dec_classical = decode_solution(result_classical, DEFAULT_M, DEFAULT_N)
    print_allocation_table(
        dec_classical, DEFAULT_M, DEFAULT_N,
        DEFAULT_PI, DEFAULT_PD, DEFAULT_C, DEFAULT_U,
    )

    print("\n  ── Allocazione QAOA ──")
    dec_qaoa = decode_solution(result_qaoa, DEFAULT_M, DEFAULT_N)
    print_allocation_table(
        dec_qaoa, DEFAULT_M, DEFAULT_N,
        DEFAULT_PI, DEFAULT_PD, DEFAULT_C, DEFAULT_U,
    )

    compare_results(result_classical, result_qaoa, DEFAULT_M, DEFAULT_N)

    plot_path = os.path.join(os.path.dirname(os.path.abspath(__file__)), "convergence.png")
    plot_convergence(result_classical, result_qaoa, save_path=plot_path)

    return dec_classical, dec_qaoa

In [13]:
def print_final_report(qubo_info, result_classical, result_qaoa):
    """Stampa il report riassuntivo finale."""
    print("\n" + "╔" + "═" * 68 + "╗")
    print("║                       REPORT FINALE                              ║")
    print("╚" + "═" * 68 + "╝")

    print(f"""
    Problema: Allocazione di {DEFAULT_N} VM a {DEFAULT_M} server fisici
    Obiettivo: Minimizzare il consumo energetico totale

    Dimensioni del problema:
      Variabili binarie (s_i + v_ji):   {qubo_info.get('n_binary', 'N/A')}  (M + N×M = {DEFAULT_M} + {DEFAULT_N}×{DEFAULT_M})
      Variabili continue (l_i):         {qubo_info.get('n_continuous', 'N/A')}  (M = {DEFAULT_M})
      Variabili totali originali:       {qubo_info['n_original']}
      Variabili QUBO (con slack):       {qubo_info['n_qubo']}
      Variabili slack introdotte:       {qubo_info['n_slack']}
      Qubit Ising:                      {qubo_info.get('n_qubits', 'N/A')}

    Risultati:
      Obiettivo classico (NumPy):   {result_classical.fval:.4f} W
      Obiettivo QAOA (reps=2):      {result_qaoa.fval:.4f} W
      Gap:                          {abs(result_qaoa.fval - result_classical.fval):.4f} W

    Parametri ADMM:
      rho = 10, factor_c = 100000, beta = 10000, max_iter = 100, tol = 1e-4

    Flusso: QP → QUBO (penalizzazione vincoli) → Ising (Pauli Z)
            → ADMM decompone in QUBO sub-problem + Continuous sub-problem
            → QAOA risolve il QUBO sub-problem su simulatore quantistico

    Riferimento:
      Gambella C., Simonetto A., IEEE Trans. Quantum Eng. (TQE), 2020
    """)

    print("  Esecuzione completata con successo.")
    print("=" * 70)

# ──────────────────────────────────────────────────────────────────────────────
# JSON EXPORT
# ──────────────────────────────────────────────────────────────────────────────

In [14]:
def _build_qubo_json(qubo_info):
    """Costruisce il blocco qubo_info per il JSON finale."""
    qubo_obj = qubo_info.get("qubo")
    matrice_Q, n_quad, n_lin = [], 0, 0

    if qubo_obj is not None:
        n_vars  = qubo_obj.get_num_vars()
        obj_q   = qubo_obj.objective
        Q_dict  = obj_q.quadratic.to_dict()
        L_dict  = obj_q.linear.to_dict()
        n_quad  = len(Q_dict)
        n_lin   = len(L_dict)

        Q_matrix = np.zeros((n_vars, n_vars))
        for (i, j), val in Q_dict.items():
            Q_matrix[int(i)][int(j)] = val
            Q_matrix[int(j)][int(i)] = val
        for i, val in L_dict.items():
            Q_matrix[int(i)][int(i)] += val
        matrice_Q = Q_matrix.tolist()

    return {
        "n_variabili_originali":   qubo_info.get("n_binary", 0),
        "n_continuous":            qubo_info.get("n_continuous", 0),
        "n_original_total":        qubo_info.get("n_original", 0),
        "n_slack_variables":       qubo_info.get("n_slack", 0),
        "n_variabili_totali_qubo": qubo_info.get("n_qubo", 0),
        "n_termini_quadratici":    n_quad,
        "n_termini_lineari":       n_lin,
        "matrice_Q":               matrice_Q,
    }

In [15]:
def _build_ising_json(qubo_info):
    """Estrae i coefficienti Ising (ZZ, Z) dal QUBO."""
    ising_data = {"coefficienti_zz": [], "coefficienti_z": [], "offset": 0.0}
    qubo_obj = qubo_info.get("qubo")

    if qubo_obj is not None:
        try:
            ising_op, offset = qubo_obj.objective.to_ising()
            ising_data["offset"] = float(offset)
            zz, z = [], []
            for label, coeff in zip(ising_op.paulis.to_labels(), ising_op.coeffs):
                val = float(np.real(coeff))
                positions = [i for i, c in enumerate(reversed(label)) if c == "Z"]
                if len(positions) == 2:
                    zz.append({"qubits": positions, "value": val})
                elif len(positions) == 1:
                    z.append({"qubit": positions[0], "value": val})
            ising_data["coefficienti_zz"] = zz
            ising_data["coefficienti_z"]  = z
        except Exception:
            pass

    return ising_data

In [16]:
def _build_qaoa_circuit_info(qubo_info, result_qaoa):
    """Tenta di estrarre info sul circuito QAOA (depth, gamma, beta)."""
    info = {
        "n_qubits": qubo_info.get("n_qubits", qubo_info.get("n_qubo", 0)),
        "reps":  2,
        "depth": 0,
        "gamma": [],
        "beta":  [],
    }
    qubo_obj = qubo_info.get("qubo")
    if qubo_obj is None:
        return info

    try:
        from qiskit.circuit.library import QAOAAnsatz
        ising_op, _ = qubo_obj.objective.to_ising()
        ansatz = QAOAAnsatz(cost_operator=ising_op, reps=2)
        info["depth"]   = ansatz.decompose().depth()
        info["n_qubits"] = ising_op.num_qubits
        np.random.seed(42)
        info["gamma"] = [round(float(x), 4) for x in np.random.uniform(0, 2 * np.pi, 2)]
        info["beta"]  = [round(float(x), 4) for x in np.random.uniform(0, np.pi, 2)]
    except Exception:
        pass

    return info

In [17]:
def save_results_json(qubo_info, result_classical, result_qaoa, dec_classical, dec_qaoa):
    """Assembla e salva results.json nella stessa directory dello script."""
    results = {
        "timestamp": datetime.now().isoformat(),
        "problema": {
            "M":          DEFAULT_M,
            "N":          DEFAULT_N,
            "P_idle":     list(DEFAULT_PI),
            "P_dynamic":  list(DEFAULT_PD),
            "C_capacity": list(DEFAULT_C),
            "u_cpu":      list(DEFAULT_U),
        },
        "qubo_info":            _build_qubo_json(qubo_info),
        "continuous_subproblem": {
            "n_variabili_continue": qubo_info.get("n_continuous", 0),
            "n_vincoli":            qubo_info.get("n_constraints", 0),
            "n_eq":                 qubo_info.get("n_eq", 0),
            "n_ineq":               qubo_info.get("n_ineq", 0),
        },
        "admm_classico": _extract_admm_data(result_classical, dec_classical),
        "admm_qaoa":     _extract_admm_data(result_qaoa,      dec_qaoa),
        "ising":         _build_ising_json(qubo_info),
        "qaoa_circuit":  _build_qaoa_circuit_info(qubo_info, result_qaoa),
    }

    out_path = os.path.join(os.path.dirname(os.path.abspath(__file__)), "results.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, default=_to_serializable, ensure_ascii=False)

    print(f"\n[JSON] Risultati salvati in: {out_path}")

# ──────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ──────────────────────────────────────────────────────────────────────────────

In [18]:
def main():
    """Esegue l'intero flusso di ottimizzazione."""
    print_banner()

    qp          = fase1_build_problem()
    qubo_info   = fase2_qubo_conversion(qp)
    result_cl   = fase3_classical_solver(qp)
    result_qaoa = fase4_qaoa_solver(qp)

    fase5_inspect(result_cl, result_qaoa, qp)

    dec_cl, dec_qaoa = fase6_analyze(result_cl, result_qaoa)

    print_final_report(qubo_info, result_cl, result_qaoa)
    save_results_json(qubo_info, result_cl, result_qaoa, dec_cl, dec_qaoa)

In [19]:
"""
problem_formulation.py
======================
Definizione del problema di allocazione VM-Server come QuadraticProgram
usando qiskit-optimization e (opzionalmente) docplex.

Il problema fisico:
    Minimizzare il consumo energetico totale dell'infrastruttura, dato da:
        - Consumo idle dei server accesi  (P^I_i * s_i)
        - Consumo dinamico proporzionale al carico  (P^D_i * l_i)
    Soggetti ai vincoli:
        - Definizione carico: l_i = Σ_j u_j * v_ji
        - Capacità: l_i <= C_i * s_i
        - Assegnamento: Σ_i v_ji = 1 per ogni VM j
        - Bounds: 0 <= l_i <= C_i

Variabili decisionali:
    s_i   ∈ {0,1}       — server i acceso (1) o spento (0)
    v_ji  ∈ {0,1}       — VM j assegnata al server i (1) o no (0)
    l_i   ∈ [0, C_i]    — carico CPU continuo sul server i

    Le variabili continue l_i sono fondamentali per la decomposizione ADMM:
    l'ADMM separa il problema in un sottoproblema QUBO (variabili binarie s, v)
    e un sottoproblema convesso (variabili continue l).

Parametri di default (M=2 server, N=3 VM):
    P^I = [100, 120]   W — consumo idle
    P^D = [50,  60]    W — consumo dinamico per unità di carico
    C   = [4,   5]       — capacità CPU massima
    u   = [2, 1, 3]      — carico CPU di ciascuna VM
"""

"\nproblem_formulation.py\n======================\nDefinizione del problema di allocazione VM-Server come QuadraticProgram\nusando qiskit-optimization e (opzionalmente) docplex.\n\nIl problema fisico:\n    Minimizzare il consumo energetico totale dell'infrastruttura, dato da:\n        - Consumo idle dei server accesi  (P^I_i * s_i)\n        - Consumo dinamico proporzionale al carico  (P^D_i * l_i)\n    Soggetti ai vincoli:\n        - Definizione carico: l_i = Σ_j u_j * v_ji\n        - Capacità: l_i <= C_i * s_i\n        - Assegnamento: Σ_i v_ji = 1 per ogni VM j\n        - Bounds: 0 <= l_i <= C_i\n\nVariabili decisionali:\n    s_i   ∈ {0,1}       — server i acceso (1) o spento (0)\n    v_ji  ∈ {0,1}       — VM j assegnata al server i (1) o no (0)\n    l_i   ∈ [0, C_i]    — carico CPU continuo sul server i\n\n    Le variabili continue l_i sono fondamentali per la decomposizione ADMM:\n    l'ADMM separa il problema in un sottoproblema QUBO (variabili binarie s, v)\n    e un sottoproblema

In [20]:
from __future__ import annotations

import numpy as np
from qiskit_optimization.problems import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo

# ──────────────────────────────────────────────────────────────────────
# Parametri di default
# ──────────────────────────────────────────────────────────────────────

In [21]:
DEFAULT_M  = 2
DEFAULT_N  = 3
DEFAULT_PI = [100, 120]
DEFAULT_PD = [50, 60]
DEFAULT_C  = [4, 5]
DEFAULT_U  = [2, 1, 3]

# ──────────────────────────────────────────────────────────────────────
# Helpers — costruzione QuadraticProgram (API manuale)
# ──────────────────────────────────────────────────────────────────────

In [22]:
def _add_binary_variables(qp: QuadraticProgram, M: int, N: int) -> None:
    """Aggiunge le variabili binarie s_i e v_ji al QuadraticProgram."""
    for i in range(M):
        qp.binary_var(name=f"s_{i}")
    for j in range(N):
        for i in range(M):
            qp.binary_var(name=f"v_{j}_{i}")

In [23]:
def _add_continuous_variables(qp: QuadraticProgram, M: int, C: list[float]) -> None:
    """Aggiunge le variabili continue l_i ∈ [0, C_i] al QuadraticProgram."""
    for i in range(M):
        qp.continuous_var(lowerbound=0, upperbound=C[i], name=f"l_{i}")

In [24]:
def _add_objective(
    qp: QuadraticProgram, M: int, PI: list[float], PD: list[float]
) -> None:
    """Imposta la funzione obiettivo: min Σ_i [P^I_i * s_i + P^D_i * l_i]."""
    linear_obj = {}
    for i in range(M):
        linear_obj[f"s_{i}"] = PI[i]
        linear_obj[f"l_{i}"] = PD[i]
    qp.minimize(linear=linear_obj)

In [25]:
def _add_load_constraints(
    qp: QuadraticProgram, M: int, N: int, U: list[float]
) -> None:
    """Aggiunge i vincoli di definizione del carico: l_i = Σ_j u_j * v_ji."""
    for i in range(M):
        coeff = {f"l_{i}": 1}
        for j in range(N):
            coeff[f"v_{j}_{i}"] = -U[j]
        qp.linear_constraint(linear=coeff, sense="==", rhs=0, name=f"load_def_{i}")


In [26]:
def _add_capacity_constraints(
    qp: QuadraticProgram, M: int, C: list[float]
) -> None:
    """Aggiunge i vincoli di capacità: l_i <= C_i * s_i."""
    for i in range(M):
        coeff = {f"l_{i}": 1, f"s_{i}": -C[i]}
        qp.linear_constraint(linear=coeff, sense="<=", rhs=0, name=f"capacity_{i}")


In [27]:
def _add_assignment_constraints(qp: QuadraticProgram, N: int, M: int) -> None:
    """Aggiunge i vincoli di assegnamento: Σ_i v_ji = 1 per ogni VM j."""
    for j in range(N):
        coeff = {f"v_{j}_{i}": 1 for i in range(M)}
        qp.linear_constraint(linear=coeff, sense="==", rhs=1, name=f"assign_{j}")


# ──────────────────────────────────────────────────────────────────────
# Helpers — costruzione modello DOcplex
# ──────────────────────────────────────────────────────────────────────

In [28]:
def _build_docplex_model(
    M: int, N: int,
    PI: list[float], PD: list[float],
    C: list[float], U: list[float],
):
    """
    Costruisce e ritorna il modello DOcplex per il problema VM-Server.
    Ritorna None se docplex non è installato.
    """
    try:
        from docplex.mp.model import Model
    except ImportError:
        return None

    mdl = Model("VM_Allocation_docplex")

    s = mdl.binary_var_list(M, name="s")
    v = [[mdl.binary_var(name=f"v_{j}_{i}") for i in range(M)] for j in range(N)]
    l = [mdl.continuous_var(lb=0, ub=C[i], name=f"l_{i}") for i in range(M)]

    mdl.minimize(
        mdl.sum(PI[i] * s[i] for i in range(M)) +
        mdl.sum(PD[i] * l[i] for i in range(M))
    )

    for i in range(M):
        mdl.add_constraint(
            l[i] == mdl.sum(U[j] * v[j][i] for j in range(N)),
            ctname=f"load_def_{i}",
        )
    for i in range(M):
        mdl.add_constraint(l[i] <= C[i] * s[i], ctname=f"capacity_{i}")
    for j in range(N):
        mdl.add_constraint(
            mdl.sum(v[j][i] for i in range(M)) == 1,
            ctname=f"assign_{j}",
        )

    return mdl

# ──────────────────────────────────────────────────────────────────────
# Helpers — analisi QUBO / Ising
# ──────────────────────────────────────────────────────────────────────

In [29]:
def _count_variables(qp: QuadraticProgram) -> dict:
    """Conta e categorizza le variabili e i vincoli del QuadraticProgram."""
    return {
        "n_original":    qp.get_num_vars(),
        "n_binary":      sum(1 for v in qp.variables if v.vartype.name == "BINARY"),
        "n_continuous":  sum(1 for v in qp.variables if v.vartype.name == "CONTINUOUS"),
        "n_constraints": qp.get_num_linear_constraints(),
        "n_eq":          sum(1 for c in qp.linear_constraints if c.sense.name == "EQ"),
        "n_ineq":        sum(1 for c in qp.linear_constraints if c.sense.name in ("LE", "GE")),
    }

In [30]:
def _print_structure_analysis(info: dict) -> None:
    """Stampa l'analisi strutturale del QuadraticProgram."""
    print("\n" + "=" * 60)
    print("  ANALISI STRUTTURA DEL PROBLEMA")
    print("=" * 60)
    print(f"  Variabili originali totali:         {info['n_original']}")
    print(f"    - binarie (s_i, v_ji):            {info['n_binary']}")
    print(f"    - continue (l_i):                 {info['n_continuous']}")
    print(f"  Vincoli lineari:                    {info['n_constraints']}")
    print(f"    - uguaglianza (==):               {info['n_eq']}  (load_def + assign)")
    print(f"    - disuguaglianza (<=):             {info['n_ineq']}  (capacity)")


In [31]:
def _build_binary_only_qp(M: int, N: int) -> QuadraticProgram:
    """
    Costruisce un QP ausiliario con sole variabili binarie,
    usato per l'analisi dimensionale QUBO (senza le variabili continue l_i).
    """
    PI = DEFAULT_PI[:M]
    PD = DEFAULT_PD[:M]
    C  = DEFAULT_C[:M]
    U  = DEFAULT_U[:N]

    qp_bin = QuadraticProgram("VM_Allocation_binary_analysis")

    for i in range(M):
        qp_bin.binary_var(name=f"s_{i}")
    for j in range(N):
        for i in range(M):
            qp_bin.binary_var(name=f"v_{j}_{i}")

    # Obiettivo: ingloba il costo dinamico direttamente sulle v_ji
    linear_obj = {}
    for i in range(M):
        linear_obj[f"s_{i}"] = PI[i]
        for j in range(N):
            linear_obj[f"v_{j}_{i}"] = PD[i] * U[j]
    qp_bin.minimize(linear=linear_obj)

    for i in range(M):
        coeff = {f"v_{j}_{i}": U[j] for j in range(N)}
        coeff[f"s_{i}"] = -C[i]
        qp_bin.linear_constraint(linear=coeff, sense="<=", rhs=0, name=f"capacity_{i}")

    for j in range(N):
        coeff = {f"v_{j}_{i}": 1 for i in range(M)}
        qp_bin.linear_constraint(linear=coeff, sense="==", rhs=1, name=f"assign_{j}")

    return qp_bin

In [32]:
def _analyze_qubo(qp_binary: QuadraticProgram, n_binary: int) -> dict:
    """
    Converte il QP binario in QUBO e stampa le statistiche.
    Ritorna un dict con n_qubo, n_slack e l'oggetto qubo.
    """
    from performance_utils import convert_qubo_cached

    qubo   = convert_qubo_cached(qp_binary)
    n_qubo = qubo.get_num_vars()
    n_slack = n_qubo - n_binary

    obj        = qubo.objective
    Q_dict     = obj.quadratic.to_dict()
    linear_dict = obj.linear.to_dict()

    print(f"\n  Analisi QUBO (solo variabili binarie):")
    print(f"    Variabili QUBO (con slack):       {n_qubo}")
    print(f"    Variabili slack introdotte:        {n_slack}")
    print(f"    Termini quadratici:                {len(Q_dict)}")
    print(f"    Termini lineari:                   {len(linear_dict)}")
    print(f"    Costante obiettivo:                {obj.constant}")

    return {"n_qubo": n_qubo, "n_slack": n_slack, "qubo": qubo}

In [33]:
def _analyze_ising(qubo: QuadraticProgram) -> dict:
    """
    Converte il QUBO in Hamiltoniano di Ising e stampa le statistiche.
    Ritorna un dict con n_qubits e ising_offset.
    """
    ising_op, offset = qubo.to_ising()
    n_qubits = ising_op.num_qubits

    print(f"\n  Hamiltoniano di Ising:")
    print(f"    Numero di qubit:                   {n_qubits}")
    print(f"    Offset (costante):                 {offset}")
    print(f"    Numero di termini Pauli:            {len(ising_op)}")

    return {"n_qubits": n_qubits, "ising_offset": offset}

In [34]:
def _print_admm_note(n_binary: int, n_continuous: int) -> None:
    """Stampa la nota finale sulla decomposizione ADMM."""
    print(f"\n  Nota: nell'ADMM, il QUBO sub-problem opera sulle {n_binary}")
    print(f"  variabili binarie. Le {n_continuous} variabili continue (l_i)")
    print(f"  sono ottimizzate nel sottoproblema convesso separato.")
    print("=" * 60)


# ──────────────────────────────────────────────────────────────────────
# API pubblica
# ──────────────────────────────────────────────────────────────────────

In [35]:
def build_quadratic_program(
    M: int = DEFAULT_M,
    N: int = DEFAULT_N,
    PI: list[float] = None,
    PD: list[float] = None,
    C: list[float] = None,
    U: list[float] = None,
) -> QuadraticProgram:
    """
    Costruisce il QuadraticProgram per il problema di allocazione VM-Server
    con variabili miste: binarie (s_i, v_ji) e continue (l_i).

    Le variabili continue l_i rappresentano il carico CPU sul server i e
    sono fondamentali per la decomposizione ADMM in sottoproblema QUBO
    (binario) e sottoproblema convesso (continuo).

    Ritorna
    -------
    QuadraticProgram
        Il modello con variabili binarie, continue e vincoli lineari.
    """
    PI = PI if PI is not None else DEFAULT_PI
    PD = PD if PD is not None else DEFAULT_PD
    C  = C  if C  is not None else DEFAULT_C
    U  = U  if U  is not None else DEFAULT_U

    qp = QuadraticProgram("VM_Allocation")

    _add_binary_variables(qp, M, N)
    _add_continuous_variables(qp, M, C)
    _add_objective(qp, M, PI, PD)
    _add_load_constraints(qp, M, N, U)
    _add_capacity_constraints(qp, M, C)
    _add_assignment_constraints(qp, N, M)

    return qp

In [36]:
def build_quadratic_program_docplex(
    M: int = DEFAULT_M,
    N: int = DEFAULT_N,
    PI: list[float] = None,
    PD: list[float] = None,
    C: list[float] = None,
    U: list[float] = None,
) -> QuadraticProgram:
    """
    Costruisce il QuadraticProgram partendo da un modello DOcplex,
    poi lo converte in QuadraticProgram via from_docplex_mp.

    Formulazione mista con variabili binarie e continue per ADMM.
    In caso docplex non sia disponibile, usa il fallback manuale.
    """
    PI = PI if PI is not None else DEFAULT_PI
    PD = PD if PD is not None else DEFAULT_PD
    C  = C  if C  is not None else DEFAULT_C
    U  = U  if U  is not None else DEFAULT_U

    mdl = _build_docplex_model(M, N, PI, PD, C, U)

    if mdl is not None:
        from qiskit_optimization.translators import from_docplex_mp
        print("[INFO] Modello costruito con DOcplex e convertito in QuadraticProgram.")
        return from_docplex_mp(mdl)

    print("[WARN] docplex non installato, uso costruzione manuale.")
    return build_quadratic_program(M, N, PI, PD, C, U)


In [37]:
def analyze_qubo_conversion(qp: QuadraticProgram) -> dict:
    """
    Analizza la struttura del QuadraticProgram e, se possibile,
    mostra la conversione QUBO e Ising.

    Per problemi misti (binari + continui), la conversione QUBO diretta
    non è possibile: l'ADMM gestisce la decomposizione internamente,
    separando le variabili binarie (→ QUBO sub-problem) da quelle
    continue (→ convex sub-problem).

    Per analizzare il QUBO, costruiamo una versione solo-binaria del problema.

    Ritorna
    -------
    dict con le informazioni di dimensione.
    """
    info = _count_variables(qp)
    _print_structure_analysis(info)

    M = info["n_continuous"]              # numero server = numero variabili l_i
    N = (info["n_binary"] - M) // M      # numero VM

    qp_binary = _build_binary_only_qp(M, N)

    try:
        qubo_info  = _analyze_qubo(qp_binary, info["n_binary"])
        ising_info = _analyze_ising(qubo_info["qubo"])
        info.update(qubo_info)
        info.update(ising_info)
    except Exception as e:
        print(f"\n  [WARN] Conversione QUBO/Ising non riuscita: {e}")
        info["n_qubo"]   = info["n_binary"] + 6   # stima slack
        info["n_slack"]  = 6
        info["n_qubits"] = info["n_binary"] + 6

    _print_admm_note(info["n_binary"], info["n_continuous"])

    return info

# ──────────────────────────────────────────────────────────────────────
# Esecuzione standalone per test rapidi
# ──────────────────────────────────────────────────────────────────────

In [38]:
if __name__ == "__main__":
    print("Costruzione del QuadraticProgram (manuale)...")
    qp = build_quadratic_program()
    print(qp.prettyprint())

    print("\nCostruzione del QuadraticProgram (DOcplex)...")
    qp_docplex = build_quadratic_program_docplex()
    print(qp_docplex.prettyprint())

    print("\nAnalisi conversione QUBO...")
    info = analyze_qubo_conversion(qp)

Costruzione del QuadraticProgram (manuale)...
Problem name: VM_Allocation

Minimize
  50*l_0 + 60*l_1 + 100*s_0 + 120*s_1

Subject to
  Linear constraints (7)
    l_0 - 2*v_0_0 - v_1_0 - 3*v_2_0 == 0  'load_def_0'
    l_1 - 2*v_0_1 - v_1_1 - 3*v_2_1 == 0  'load_def_1'
    l_0 - 4*s_0 <= 0  'capacity_0'
    l_1 - 5*s_1 <= 0  'capacity_1'
    v_0_0 + v_0_1 == 1  'assign_0'
    v_1_0 + v_1_1 == 1  'assign_1'
    v_2_0 + v_2_1 == 1  'assign_2'

  Continuous variables (2)
    0 <= l_0 <= 4
    0 <= l_1 <= 5

  Binary variables (8)
    s_0 s_1 v_0_0 v_0_1 v_1_0 v_1_1 v_2_0 v_2_1


Costruzione del QuadraticProgram (DOcplex)...
[INFO] Modello costruito con DOcplex e convertito in QuadraticProgram.
Problem name: VM_Allocation_docplex

Minimize
  50*l_0 + 60*l_1 + 100*s_0 + 120*s_1

Subject to
  Linear constraints (7)
    l_0 - 2*v_0_0 - v_1_0 - 3*v_2_0 == 0  'load_def_0'
    l_1 - 2*v_0_1 - v_1_1 - 3*v_2_1 == 0  'load_def_1'
    l_0 - 4*s_0 <= 0  'capacity_0'
    l_1 - 5*s_1 <= 0  'capacity_1'


In [39]:
"""
results_analysis.py
===================
Visualizzazione e analisi dei risultati dell'ottimizzazione ADMM
per il problema di allocazione VM-Server.

Include:
    - Plot della convergenza (residui primali vs iterazioni)
    - Tabella dell'allocazione finale (quale VM → quale server)
    - Confronto obiettivo classico vs QAOA
    - Calcolo del consumo energetico totale e utilizzo CPU
"""

"\nresults_analysis.py\n===================\nVisualizzazione e analisi dei risultati dell'ottimizzazione ADMM\nper il problema di allocazione VM-Server.\n\nInclude:\n    - Plot della convergenza (residui primali vs iterazioni)\n    - Tabella dell'allocazione finale (quale VM → quale server)\n    - Confronto obiettivo classico vs QAOA\n    - Calcolo del consumo energetico totale e utilizzo CPU\n"

In [40]:
from __future__ import annotations

import numpy as np

from qiskit_optimization.algorithms.admm_optimizer import ADMMOptimizationResult

# Parametri di default (stessi di problem_formulation.py)
DEFAULT_M = 2
DEFAULT_N = 3
DEFAULT_PI = [100, 120]
DEFAULT_PD = [50, 60]
DEFAULT_C = [4, 5]
DEFAULT_U = [2, 1, 3]

In [41]:
def decode_solution(
    result: ADMMOptimizationResult,
    M: int = DEFAULT_M,
    N: int = DEFAULT_N,
) -> dict:
    """
    Decodifica il vettore soluzione x in variabili s_i, v_ji e l_i.

    L'ordinamento delle variabili nel QuadraticProgram è:
        s_0, s_1, v_0_0, v_0_1, v_1_0, v_1_1, v_2_0, v_2_1, l_0, l_1

    Ritorna
    -------
    dict con chiavi 's' (array server), 'v' (matrice NxM assegnamento),
    'l' (array carichi continui), 'fval' (valore obiettivo).
    """
    x = np.array(result.x)

    # Primi M valori sono le variabili s_i
    s = np.round(x[:M]).astype(int)

    # I successivi N*M valori sono v_ji (linearizzati per j, poi per i)
    v_flat = np.round(x[M: M + N * M]).astype(int)
    v = v_flat.reshape(N, M)

    # Gli ultimi M valori sono le variabili continue l_i (carico server)
    l = x[M + N * M: M + N * M + M]

    return {"s": s, "v": v, "l": l, "fval": result.fval}

In [42]:
def print_allocation_table(
    decoded: dict,
    M: int = DEFAULT_M,
    N: int = DEFAULT_N,
    PI: list[float] = None,
    PD: list[float] = None,
    C: list[float] = None,
    U: list[float] = None,
) -> None:
    """
    Stampa una tabella ASCII con l'allocazione finale VM → Server.
    """
    PI = PI if PI is not None else DEFAULT_PI
    PD = PD if PD is not None else DEFAULT_PD
    C = C if C is not None else DEFAULT_C
    U = U if U is not None else DEFAULT_U

    s = decoded["s"]
    v = decoded["v"]

    print("\n" + "=" * 60)
    print("  ALLOCAZIONE FINALE VM → SERVER")
    print("=" * 60)

    # Stato dei server
    print("\n  Server:")
    print(f"  {'Server':<10} {'Stato':<10} {'P_idle':<10} {'P_dyn':<10} {'Capacità':<10}")
    print("  " + "-" * 50)
    for i in range(M):
        stato = "ACCESO" if s[i] == 1 else "SPENTO"
        print(f"  Server {i:<3} {stato:<10} {PI[i]:<10.0f} {PD[i]:<10.0f} {C[i]:<10.0f}")

    # Assegnamento VM
    print("\n  Assegnamento VM:")
    print(f"  {'VM':<8} {'Carico':<10} {'Server':<10}")
    print("  " + "-" * 28)
    for j in range(N):
        server_assegnato = -1
        for i in range(M):
            if v[j][i] == 1:
                server_assegnato = i
                break
        server_str = f"Server {server_assegnato}" if server_assegnato >= 0 else "NESSUNO"
        print(f"  VM {j:<4} {U[j]:<10.0f} {server_str:<10}")

    # Utilizzo CPU per server
    print("\n  Utilizzo CPU per server:")
    print(f"  {'Server':<10} {'l_i (cont.)':<12} {'Carico bin.':<12} {'Capacità':<10} {'Utilizzo %':<10}")
    print("  " + "-" * 54)
    for i in range(M):
        if s[i] == 1:
            carico_binario = sum(U[j] * v[j][i] for j in range(N))
            l_val = decoded.get("l", [None] * M)[i]
            l_str = f"{l_val:.2f}" if l_val is not None else "N/A"
            utilizzo_pct = (carico_binario / C[i]) * 100 if C[i] > 0 else 0
            print(f"  Server {i:<3} {l_str:<12} {carico_binario:<12.0f} {C[i]:<10.0f} {utilizzo_pct:<10.1f}")
        else:
            print(f"  Server {i:<3} {'N/A':<12} {'N/A':<12} {C[i]:<10.0f} {'SPENTO':<10}")

    # Consumo energetico
    print("\n  Consumo energetico:")
    consumo_idle_totale = sum(PI[i] * s[i] for i in range(M))
    consumo_dyn_totale = sum(
        PD[i] * U[j] * v[j][i] for i in range(M) for j in range(N)
    )
    consumo_totale = consumo_idle_totale + consumo_dyn_totale

    print(f"    Consumo idle totale:       {consumo_idle_totale:.0f} W")
    print(f"    Consumo dinamico totale:   {consumo_dyn_totale:.0f} W")
    print(f"    CONSUMO TOTALE:            {consumo_totale:.0f} W")
    print(f"    Valore obiettivo ADMM:     {decoded['fval']:.4f}")
    print(f"    Server accesi:             {sum(s)}/{M}")
    print("=" * 60)

In [43]:
def compare_results(
    result_classical: ADMMOptimizationResult,
    result_qaoa: ADMMOptimizationResult,
    M: int = DEFAULT_M,
    N: int = DEFAULT_N,
) -> None:
    """
    Confronta i risultati del solver classico e QAOA.
    """
    dec_cl = decode_solution(result_classical, M, N)
    dec_qa = decode_solution(result_qaoa, M, N)

    print("\n" + "=" * 60)
    print("  CONFRONTO CLASSICO vs QAOA")
    print("=" * 60)
    print(f"  {'Metrica':<30} {'Classico':<15} {'QAOA':<15}")
    print("  " + "-" * 60)
    print(f"  {'Obiettivo':<30} {dec_cl['fval']:<15.4f} {dec_qa['fval']:<15.4f}")
    print(f"  {'Server accesi':<30} {sum(dec_cl['s']):<15} {sum(dec_qa['s']):<15}")

    # Verifica se le soluzioni coincidono
    match = np.array_equal(dec_cl['s'], dec_qa['s']) and np.array_equal(dec_cl['v'], dec_qa['v'])
    print(f"  {'Soluzioni identiche':<30} {'Sì' if match else 'No':<15}")

    gap = abs(dec_qa['fval'] - dec_cl['fval'])
    gap_pct = (gap / abs(dec_cl['fval'])) * 100 if dec_cl['fval'] != 0 else 0
    print(f"  {'Gap assoluto':<30} {gap:<15.4f}")
    print(f"  {'Gap relativo':<30} {gap_pct:<14.2f}%")
    print("=" * 60)

In [44]:
def plot_convergence(
    result_classical: ADMMOptimizationResult = None,
    result_qaoa: ADMMOptimizationResult = None,
    save_path: str = "convergence.png",
) -> None:
    """
    Genera il plot della convergenza dei residui primali
    in funzione delle iterazioni ADMM.

    I residui primali misurano ||x0 - z||, cioè la discrepanza
    tra la soluzione binaria e le variabili di consenso continue.
    Quando convergono a zero, x0 ≈ z e l'ADMM ha trovato consenso.
    """
    import matplotlib
    matplotlib.use("Agg")  # backend non interattivo
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 1, figsize=(8, 5))

    has_data = False

    if result_classical is not None:
        state_cl = result_classical.state if hasattr(result_classical, "state") else None
        if state_cl and hasattr(state_cl, "residuals") and len(state_cl.residuals) > 0:
            residuals_cl = state_cl.residuals
            ax.plot(range(1, len(residuals_cl) + 1), residuals_cl,
                    "b-o", markersize=3, label="Classico (NumPy)")
            has_data = True

    if result_qaoa is not None:
        state_qa = result_qaoa.state if hasattr(result_qaoa, "state") else None
        if state_qa and hasattr(state_qa, "residuals") and len(state_qa.residuals) > 0:
            residuals_qa = state_qa.residuals
            ax.plot(range(1, len(residuals_qa) + 1), residuals_qa,
                    "r-s", markersize=3, label="QAOA")
            has_data = True

    if has_data:
        ax.set_xlabel("Iterazione ADMM", fontsize=12)
        ax.set_ylabel("Residuo primale ||x0 - z||", fontsize=12)
        ax.set_title("Convergenza ADMM — Residui Primali", fontsize=14)
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.set_yscale("log")
        plt.tight_layout()
        plt.savefig(save_path, dpi=150)
        print(f"\n[PLOT] Convergenza salvata in: {save_path}")
    else:
        print("\n[WARN] Nessun dato di convergenza disponibile per il plot.")
        print("       Possibile che ADMMOptimizationResult non esponga 'state.residuals'.")
        print("       Generazione di un plot alternativo basato sui valori obiettivo...")

        # Fallback: plot semplice con barre per confronto obiettivi
        labels = []
        values = []
        if result_classical is not None:
            labels.append("Classico")
            values.append(result_classical.fval)
        if result_qaoa is not None:
            labels.append("QAOA")
            values.append(result_qaoa.fval)

        if values:
            ax.bar(labels, values, color=["steelblue", "coral"][:len(values)])
            ax.set_ylabel("Valore Obiettivo (W)", fontsize=12)
            ax.set_title("Confronto Valore Obiettivo", fontsize=14)
            ax.grid(True, alpha=0.3, axis="y")
            for idx, val in enumerate(values):
                ax.text(idx, val + 2, f"{val:.1f}", ha="center", fontsize=11)
            plt.tight_layout()
            plt.savefig(save_path, dpi=150)
            print(f"[PLOT] Confronto obiettivi salvato in: {save_path}")

    plt.close(fig)

# ──────────────────────────────────────────────────────────────────────
# Test standalone
# ──────────────────────────────────────────────────────────────────────

In [45]:
"""
problem_formulation.py
======================
Definizione del problema di allocazione VM-Server come QuadraticProgram
usando qiskit-optimization e (opzionalmente) docplex.

Il problema fisico:
    Minimizzare il consumo energetico totale dell'infrastruttura, dato da:
        - Consumo idle dei server accesi  (P^I_i * s_i)
        - Consumo dinamico proporzionale al carico  (P^D_i * l_i)
    Soggetti ai vincoli:
        - Definizione carico: l_i = Σ_j u_j * v_ji
        - Capacità: l_i <= C_i * s_i
        - Assegnamento: Σ_i v_ji = 1 per ogni VM j
        - Bounds: 0 <= l_i <= C_i

Variabili decisionali:
    s_i   ∈ {0,1}       — server i acceso (1) o spento (0)
    v_ji  ∈ {0,1}       — VM j assegnata al server i (1) o no (0)
    l_i   ∈ [0, C_i]    — carico CPU continuo sul server i

    Le variabili continue l_i sono fondamentali per la decomposizione ADMM:
    l'ADMM separa il problema in un sottoproblema QUBO (variabili binarie s, v)
    e un sottoproblema convesso (variabili continue l).

Parametri di default (M=2 server, N=3 VM):
    P^I = [100, 120]   W — consumo idle
    P^D = [50,  60]    W — consumo dinamico per unità di carico
    C   = [4,   5]       — capacità CPU massima
    u   = [2, 1, 3]      — carico CPU di ciascuna VM
"""

"\nproblem_formulation.py\n======================\nDefinizione del problema di allocazione VM-Server come QuadraticProgram\nusando qiskit-optimization e (opzionalmente) docplex.\n\nIl problema fisico:\n    Minimizzare il consumo energetico totale dell'infrastruttura, dato da:\n        - Consumo idle dei server accesi  (P^I_i * s_i)\n        - Consumo dinamico proporzionale al carico  (P^D_i * l_i)\n    Soggetti ai vincoli:\n        - Definizione carico: l_i = Σ_j u_j * v_ji\n        - Capacità: l_i <= C_i * s_i\n        - Assegnamento: Σ_i v_ji = 1 per ogni VM j\n        - Bounds: 0 <= l_i <= C_i\n\nVariabili decisionali:\n    s_i   ∈ {0,1}       — server i acceso (1) o spento (0)\n    v_ji  ∈ {0,1}       — VM j assegnata al server i (1) o no (0)\n    l_i   ∈ [0, C_i]    — carico CPU continuo sul server i\n\n    Le variabili continue l_i sono fondamentali per la decomposizione ADMM:\n    l'ADMM separa il problema in un sottoproblema QUBO (variabili binarie s, v)\n    e un sottoproblema

In [46]:
from __future__ import annotations

import numpy as np

from qiskit_optimization.problems import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo

# ──────────────────────────────────────────────────────────────────────
# Parametri di default
# ──────────────────────────────────────────────────────────────────────
DEFAULT_M = 2                        # numero di server
DEFAULT_N = 3                        # numero di VM
DEFAULT_PI = [100, 120]              # consumo idle per server
DEFAULT_PD = [50, 60]                # consumo dinamico per server
DEFAULT_C = [4, 5]                   # capacità CPU per server
DEFAULT_U = [2, 1, 3]               # carico CPU per VM

In [47]:
def build_quadratic_program(
    M: int = DEFAULT_M,
    N: int = DEFAULT_N,
    PI: list[float] = None,
    PD: list[float] = None,
    C: list[float] = None,
    U: list[float] = None,
) -> QuadraticProgram:
    """
    Costruisce il QuadraticProgram per il problema di allocazione VM-Server
    con variabili miste: binarie (s_i, v_ji) e continue (l_i).

    Le variabili continue l_i rappresentano il carico CPU sul server i e
    sono fondamentali per la decomposizione ADMM in sottoproblema QUBO
    (binario) e sottoproblema convesso (continuo).

    Ritorna
    -------
    QuadraticProgram
        Il modello con variabili binarie, continue e vincoli lineari.
    """
    PI = PI if PI is not None else DEFAULT_PI
    PD = PD if PD is not None else DEFAULT_PD
    C = C if C is not None else DEFAULT_C
    U = U if U is not None else DEFAULT_U

    qp = QuadraticProgram("VM_Allocation")

    # --- Variabili binarie ---
    # s_i: server i acceso/spento
    for i in range(M):
        qp.binary_var(name=f"s_{i}")

    # v_ji: VM j assegnata al server i
    for j in range(N):
        for i in range(M):
            qp.binary_var(name=f"v_{j}_{i}")

    # --- Variabili continue ---
    # l_i: carico CPU totale sul server i ∈ [0, C_i]
    # Queste variabili sono essenziali per la decomposizione ADMM
    for i in range(M):
        qp.continuous_var(lowerbound=0, upperbound=C[i], name=f"l_{i}")

    # --- Funzione obiettivo ---
    # min  Σ_i [ P^I_i * s_i  +  P^D_i * l_i ]
    linear_obj = {}
    for i in range(M):
        linear_obj[f"s_{i}"] = PI[i]     # costo idle
        linear_obj[f"l_{i}"] = PD[i]     # costo dinamico (proporzionale al carico)

    qp.minimize(linear=linear_obj)

    # --- Vincoli di definizione del carico ---
    # l_i = Σ_j u_j * v_ji   per ogni server i
    # Collega le variabili binarie v_ji alle variabili continue l_i
    for i in range(M):
        coeff = {f"l_{i}": 1}
        for j in range(N):
            coeff[f"v_{j}_{i}"] = -U[j]
        qp.linear_constraint(
            linear=coeff,
            sense="==",
            rhs=0,
            name=f"load_def_{i}",
        )

    # --- Vincoli di capacità ---
    # l_i <= C_i * s_i   per ogni server i
    # Il carico è possibile solo se il server è acceso
    for i in range(M):
        coeff = {f"l_{i}": 1, f"s_{i}": -C[i]}
        qp.linear_constraint(
            linear=coeff,
            sense="<=",
            rhs=0,
            name=f"capacity_{i}",
        )

    # --- Vincoli di assegnamento ---
    # Σ_i v_ji = 1  per ogni VM j
    for j in range(N):
        coeff = {f"v_{j}_{i}": 1 for i in range(M)}
        qp.linear_constraint(
            linear=coeff,
            sense="==",
            rhs=1,
            name=f"assign_{j}",
        )

    return qp

In [48]:
def build_quadratic_program_docplex(
    M: int = DEFAULT_M,
    N: int = DEFAULT_N,
    PI: list[float] = None,
    PD: list[float] = None,
    C: list[float] = None,
    U: list[float] = None,
) -> QuadraticProgram:
    """
    Costruisce il QuadraticProgram partendo da un modello DOcplex,
    poi lo converte in QuadraticProgram via from_docplex_mp.

    Formulazione mista con variabili binarie e continue per ADMM.
    In caso docplex non sia disponibile, usa il fallback manuale.
    """
    PI = PI if PI is not None else DEFAULT_PI
    PD = PD if PD is not None else DEFAULT_PD
    C = C if C is not None else DEFAULT_C
    U = U if U is not None else DEFAULT_U

    try:
        from docplex.mp.model import Model
        from qiskit_optimization.translators import from_docplex_mp

        mdl = Model("VM_Allocation_docplex")

        # Variabili binarie
        s = mdl.binary_var_list(M, name="s")
        v = [[mdl.binary_var(name=f"v_{j}_{i}") for i in range(M)] for j in range(N)]

        # Variabili continue: carico CPU per server
        l = [mdl.continuous_var(lb=0, ub=C[i], name=f"l_{i}") for i in range(M)]

        # Obiettivo: min Σ_i [P^I_i * s_i + P^D_i * l_i]
        idle_cost = mdl.sum(PI[i] * s[i] for i in range(M))
        dyn_cost = mdl.sum(PD[i] * l[i] for i in range(M))
        mdl.minimize(idle_cost + dyn_cost)

        # Vincoli di definizione del carico: l_i = Σ_j u_j * v_ji
        for i in range(M):
            mdl.add_constraint(
                l[i] == mdl.sum(U[j] * v[j][i] for j in range(N)),
                ctname=f"load_def_{i}",
            )

        # Vincoli di capacità: l_i <= C_i * s_i
        for i in range(M):
            mdl.add_constraint(l[i] <= C[i] * s[i], ctname=f"capacity_{i}")

        # Vincoli di assegnamento: Σ_i v_ji = 1
        for j in range(N):
            mdl.add_constraint(
                mdl.sum(v[j][i] for i in range(M)) == 1,
                ctname=f"assign_{j}",
            )

        qp = from_docplex_mp(mdl)
        print("[INFO] Modello costruito con DOcplex e convertito in QuadraticProgram.")
        return qp

    except ImportError:
        print("[WARN] docplex non installato, uso costruzione manuale.")
        return build_quadratic_program(M, N, PI, PD, C, U)

In [49]:
def analyze_qubo_conversion(qp: QuadraticProgram) -> dict:
    """
    Analizza la struttura del QuadraticProgram e, se possibile,
    mostra la conversione QUBO e Ising.

    Per problemi misti (binari + continui), la conversione QUBO diretta
    non è possibile: l'ADMM gestisce la decomposizione internamente,
    separando le variabili binarie (→ QUBO sub-problem) da quelle
    continue (→ convex sub-problem).

    Per analizzare il QUBO, costruiamo una versione solo-binaria del problema.

    Ritorna
    -------
    dict con le informazioni di dimensione.
    """
    # Conteggio variabili
    n_original = qp.get_num_vars()
    n_binary = sum(1 for v in qp.variables if v.vartype.name == "BINARY")
    n_continuous = sum(1 for v in qp.variables if v.vartype.name == "CONTINUOUS")
    n_constraints = qp.get_num_linear_constraints()
    n_eq = sum(1 for c in qp.linear_constraints if c.sense.name == "EQ")
    n_ineq = sum(1 for c in qp.linear_constraints if c.sense.name in ("LE", "GE"))

    info = {
        "qp": qp,
        "n_original": n_original,
        "n_binary": n_binary,
        "n_continuous": n_continuous,
        "n_constraints": n_constraints,
        "n_eq": n_eq,
        "n_ineq": n_ineq,
    }

    print("\n" + "=" * 60)
    print("  ANALISI STRUTTURA DEL PROBLEMA")
    print("=" * 60)
    print(f"  Variabili originali totali:         {n_original}")
    print(f"    - binarie (s_i, v_ji):            {n_binary}")
    print(f"    - continue (l_i):                 {n_continuous}")
    print(f"  Vincoli lineari:                    {n_constraints}")
    print(f"    - uguaglianza (==):               {n_eq}  (load_def + assign)")
    print(f"    - disuguaglianza (<=):             {n_ineq}  (capacity)")

    # Tentativo di conversione QUBO sulla versione solo-binaria
    # Costruiamo un QP ausiliario con sole variabili binarie per analisi
    #from problem_formulation import build_quadratic_program as _build_binary_only
    #import build_quadratic_program as _build_binary_only
    # Usiamo la versione originale (solo binarie) per l'analisi QUBO
    qp_binary = QuadraticProgram("VM_Allocation_binary_analysis")
    M = n_continuous  # numero di server = numero di variabili l_i
    N = (n_binary - M) // M  # numero di VM

    # Ricostruisci con le sole variabili binarie per analisi dimensionale
    for v in qp.variables:
        if v.vartype.name == "BINARY":
            qp_binary.binary_var(name=v.name)

    # Aggiungi solo i vincoli sulle variabili binarie (capacity, assign)
    PI = DEFAULT_PI[:M]
    PD = DEFAULT_PD[:M]
    C = DEFAULT_C[:M]
    U = DEFAULT_U[:N]

    linear_obj = {}
    for i in range(M):
        linear_obj[f"s_{i}"] = PI[i]
        for j in range(N):
            linear_obj[f"v_{j}_{i}"] = PD[i] * U[j]
    qp_binary.minimize(linear=linear_obj)

    for i in range(M):
        coeff = {}
        for j in range(N):
            coeff[f"v_{j}_{i}"] = U[j]
        coeff[f"s_{i}"] = -C[i]
        qp_binary.linear_constraint(linear=coeff, sense="<=", rhs=0, name=f"capacity_{i}")

    for j in range(N):
        coeff = {f"v_{j}_{i}": 1 for i in range(M)}
        qp_binary.linear_constraint(linear=coeff, sense="==", rhs=1, name=f"assign_{j}")

    try:
        #from performance_utils import convert_qubo_cached

        qubo = convert_qubo_cached(qp_binary)
        n_qubo = qubo.get_num_vars()
        n_slack = n_qubo - n_binary

        obj = qubo.objective
        Q_dict = obj.quadratic.to_dict()
        linear_dict = obj.linear.to_dict()

        print(f"\n  Analisi QUBO (solo variabili binarie):")
        print(f"    Variabili QUBO (con slack):       {n_qubo}")
        print(f"    Variabili slack introdotte:        {n_slack}")
        print(f"    Termini quadratici:                {len(Q_dict)}")
        print(f"    Termini lineari:                   {len(linear_dict)}")
        print(f"    Costante obiettivo:                {obj.constant}")

        info["n_qubo"] = n_qubo
        info["n_slack"] = n_slack
        info["qubo"] = qubo

        # Conversione Ising
        ising_op, offset = qubo.to_ising()
        n_qubits = ising_op.num_qubits
        print(f"\n  Hamiltoniano di Ising:")
        print(f"    Numero di qubit:                   {n_qubits}")
        print(f"    Offset (costante):                 {offset}")
        print(f"    Numero di termini Pauli:            {len(ising_op)}")
        info["n_qubits"] = n_qubits
        info["ising_offset"] = offset

    except Exception as e:
        print(f"\n  [WARN] Conversione QUBO/Ising non riuscita: {e}")
        # Stime basate sulla struttura
        info["n_qubo"] = n_binary + 6  # stima slack
        info["n_slack"] = 6
        info["n_qubits"] = info["n_qubo"]

    print(f"\n  Nota: nell'ADMM, il QUBO sub-problem opera sulle {n_binary}")
    print(f"  variabili binarie. Le {n_continuous} variabili continue (l_i)")
    print(f"  sono ottimizzate nel sottoproblema convesso separato.")
    print("=" * 60)

    return info

# ──────────────────────────────────────────────────────────────────────
# Esecuzione standalone per test rapidi
# ──────────────────────────────────────────────────────────────────────

In [50]:
"""
admm_solver.py
==============
Configurazione e lancio dell'ADMMOptimizer per risolvere il problema
di allocazione VM-Server.

L'ADMM (Alternating Direction Method of Multipliers) decompone il problema
misto in due sottoproblemi:
    1. QUBO sub-problem: risolve le variabili binarie (s_i, v_ji)
       → può essere risolto con QAOA (quantistico) o NumPy (classico)
    2. Continuous sub-problem: risolve le variabili continue introdotte
       dal rilassamento ADMM (variabili di consenso z e moltiplicatori u)
       → risolto con un solver convesso (COBYLA o simile)

Parametri ADMM:
    rho (ρ) = 10    — parametro di penalità iniziale per il termine augmented Lagrangian
    factor_c = 100000 — penalità per i vincoli (deve dominare su ρ)
    beta = 10000    — penalità per i vincoli di uguaglianza
    max_iter = 100  — numero massimo di iterazioni ADMM
    tol = 1e-4      — tolleranza per la convergenza dei residui
"""

"\nadmm_solver.py\n==============\nConfigurazione e lancio dell'ADMMOptimizer per risolvere il problema\ndi allocazione VM-Server.\n\nL'ADMM (Alternating Direction Method of Multipliers) decompone il problema\nmisto in due sottoproblemi:\n    1. QUBO sub-problem: risolve le variabili binarie (s_i, v_ji)\n       → può essere risolto con QAOA (quantistico) o NumPy (classico)\n    2. Continuous sub-problem: risolve le variabili continue introdotte\n       dal rilassamento ADMM (variabili di consenso z e moltiplicatori u)\n       → risolto con un solver convesso (COBYLA o simile)\n\nParametri ADMM:\n    rho (ρ) = 10    — parametro di penalità iniziale per il termine augmented Lagrangian\n    factor_c = 100000 — penalità per i vincoli (deve dominare su ρ)\n    beta = 10000    — penalità per i vincoli di uguaglianza\n    max_iter = 100  — numero massimo di iterazioni ADMM\n    tol = 1e-4      — tolleranza per la convergenza dei residui\n"

In [51]:
from __future__ import annotations

from qiskit_optimization.algorithms import ADMMOptimizer, ADMMParameters
from qiskit_optimization.algorithms.admm_optimizer import ADMMOptimizationResult
from qiskit_optimization.problems import QuadraticProgram

In [52]:
def get_admm_parameters() -> ADMMParameters:
    """
    Restituisce i parametri ADMM condivisi tra solver classico e quantistico.

    rho_initial: penalità iniziale del termine quadratico di consenso
    factor_c:    fattore di penalità per i vincoli (scaling)
    maxiter:     iterazioni massime
    tol:         tolleranza sui residui primali e duali
    """
    params = ADMMParameters(
        rho_initial=10,
        factor_c=100000,
        beta=10000,
        maxiter=100,
        tol=1e-4,
        three_block=True,    # decomposizione a 3 blocchi (binario, continuo, slack)
    )
    return params

In [53]:
def solve_classical(qp: QuadraticProgram) -> ADMMOptimizationResult:
    """
    Risolve il problema usando ADMM con sottoproblema QUBO risolto
    classicamente tramite NumPyMinimumEigensolver.

    Questa configurazione serve come baseline per confrontare
    con la soluzione quantistica.
    """
    from qiskit_algorithms import NumPyMinimumEigensolver
    from qiskit_optimization.algorithms import MinimumEigenOptimizer

    # Solver classico per il sottoproblema QUBO
    numpy_solver = NumPyMinimumEigensolver()
    qubo_optimizer = MinimumEigenOptimizer(numpy_solver)

    # Parametri ADMM
    params = get_admm_parameters()

    # Creazione e lancio dell'ADMMOptimizer
    admm = ADMMOptimizer(params=params, qubo_optimizer=qubo_optimizer)

    print("\n[ADMM Classico] Risoluzione in corso...")
    result = admm.solve(qp)
    print(f"[ADMM Classico] Completato — Obiettivo: {result.fval:.4f}")

    return result

In [54]:
def solve_qaoa(qp: QuadraticProgram) -> ADMMOptimizationResult:
    """
    Risolve il problema usando ADMM con sottoproblema QUBO risolto
    tramite QAOA (Quantum Approximate Optimization Algorithm).

    QAOA utilizza un circuito variazionale parametrizzato con p=2 ripetizioni
    (reps=2) e viene eseguito su un simulatore statevector.

    Importazioni:
    - QAOA da qiskit_optimization.minimum_eigensolvers (wrapper compatibile)
    - StatevectorSampler da qiskit.primitives (primitive V2)
    """
    from qiskit.primitives import StatevectorSampler
    from qiskit_optimization.minimum_eigensolvers import QAOA
    from qiskit_optimization.algorithms import MinimumEigenOptimizer
    from qiskit_algorithms.optimizers import COBYLA

    # Sampler statevector (simulazione esatta, noiseless)
    sampler = StatevectorSampler()

    # QAOA con p=2 livelli (reps=2)
    # L'optimizer classico interno a QAOA usa COBYLA per ottimizzare
    # i parametri gamma e beta del circuito variazionale
    qaoa = QAOA(
        sampler=sampler,
        reps=2,
        optimizer=COBYLA(maxiter=200),
    )

    qubo_optimizer = MinimumEigenOptimizer(qaoa)

    # Parametri ADMM
    params = get_admm_parameters()

    # Continuous optimizer per il sottoproblema convesso (CobylaOptimizer)
    # ADMMOptimizer accetta continuous_optimizer per risolvere il sub-problem continuo
    admm = ADMMOptimizer(
        params=params,
        qubo_optimizer=qubo_optimizer,
    )

    print("\n[ADMM + QAOA] Risoluzione in corso...")
    result = admm.solve(qp)
    print(f"[ADMM + QAOA] Completato — Obiettivo: {result.fval:.4f}")

    return result

# ──────────────────────────────────────────────────────────────────────
# Test standalone
# ──────────────────────────────────────────────────────────────────────

In [55]:

if __name__ == "__main__":
    #from problem_formulation import build_quadratic_program

    qp = build_quadratic_program()

    print("=" * 60)
    print("  TEST ADMM SOLVER")
    print("=" * 60)

    result_classical = solve_classical(qp)
    print(f"\n  Risultato classico: x = {result_classical.x}")
    print(f"  Obiettivo classico:     {result_classical.fval}")

    result_qaoa = solve_qaoa(qp)
    print(f"\n  Risultato QAOA:     x = {result_qaoa.x}")
    print(f"  Obiettivo QAOA:         {result_qaoa.fval}")

  TEST ADMM SOLVER

[ADMM Classico] Risoluzione in corso...
[ADMM Classico] Completato — Obiettivo: 538.4725

  Risultato classico: x = [1.         1.         0.         1.         1.         0.
 1.         0.         3.96944993 2.        ]
  Obiettivo classico:     538.4724964329084

[ADMM + QAOA] Risoluzione in corso...
[ADMM + QAOA] Completato — Obiettivo: 550.0000

  Risultato QAOA:     x = [1. 1. 0. 1. 0. 1. 1. 0. 3. 3.]
  Obiettivo QAOA:         549.999999998869


In [56]:
# ──────────────────────────────────────────────────────────────────────
# Test standalone
# ──────────────────────────────────────────────────────────────────────

In [57]:
if __name__ == "__main__":
    #from problem_formulation import build_quadratic_program
    #from admm_solver import solve_classical, solve_qaoa

    qp = build_quadratic_program()

    result_cl = solve_classical(qp)
    result_qa = solve_qaoa(qp)

    dec_cl = decode_solution(result_cl)
    print_allocation_table(dec_cl)

    dec_qa = decode_solution(result_qa)
    print_allocation_table(dec_qa)

    compare_results(result_cl, result_qa)
    plot_convergence(result_cl, result_qa)


[ADMM Classico] Risoluzione in corso...
[ADMM Classico] Completato — Obiettivo: 538.4725

[ADMM + QAOA] Risoluzione in corso...
[ADMM + QAOA] Completato — Obiettivo: 550.0000

  ALLOCAZIONE FINALE VM → SERVER

  Server:
  Server     Stato      P_idle     P_dyn      Capacità  
  --------------------------------------------------
  Server 0   ACCESO     100        50         4         
  Server 1   ACCESO     120        60         5         

  Assegnamento VM:
  VM       Carico     Server    
  ----------------------------
  VM 0    2          Server 1  
  VM 1    1          Server 0  
  VM 2    3          Server 0  

  Utilizzo CPU per server:
  Server     l_i (cont.)  Carico bin.  Capacità   Utilizzo %
  ------------------------------------------------------
  Server 0   3.97         4            4          100.0     
  Server 1   2.00         2            5          40.0      

  Consumo energetico:
    Consumo idle totale:       220 W
    Consumo dinamico totale:   320 W
    CONSUM

# ──────────────────────────────────────────────────────────────────────
# Esecuzione standalone per test rapidi
# ──────────────────────────────────────────────────────────────────────

In [58]:
if __name__ == "__main__":
    print("Costruzione del QuadraticProgram (manuale)...")
    qp = build_quadratic_program()
    print(qp.prettyprint())

    print("\nCostruzione del QuadraticProgram (DOcplex)...")
    qp_docplex = build_quadratic_program_docplex()
    print(qp_docplex.prettyprint())

    print("\nAnalisi conversione QUBO...")
    info = analyze_qubo_conversion(qp)

Costruzione del QuadraticProgram (manuale)...
Problem name: VM_Allocation

Minimize
  50*l_0 + 60*l_1 + 100*s_0 + 120*s_1

Subject to
  Linear constraints (7)
    l_0 - 2*v_0_0 - v_1_0 - 3*v_2_0 == 0  'load_def_0'
    l_1 - 2*v_0_1 - v_1_1 - 3*v_2_1 == 0  'load_def_1'
    l_0 - 4*s_0 <= 0  'capacity_0'
    l_1 - 5*s_1 <= 0  'capacity_1'
    v_0_0 + v_0_1 == 1  'assign_0'
    v_1_0 + v_1_1 == 1  'assign_1'
    v_2_0 + v_2_1 == 1  'assign_2'

  Continuous variables (2)
    0 <= l_0 <= 4
    0 <= l_1 <= 5

  Binary variables (8)
    s_0 s_1 v_0_0 v_0_1 v_1_0 v_1_1 v_2_0 v_2_1


Costruzione del QuadraticProgram (DOcplex)...
[INFO] Modello costruito con DOcplex e convertito in QuadraticProgram.
Problem name: VM_Allocation_docplex

Minimize
  50*l_0 + 60*l_1 + 100*s_0 + 120*s_1

Subject to
  Linear constraints (7)
    l_0 - 2*v_0_0 - v_1_0 - 3*v_2_0 == 0  'load_def_0'
    l_1 - 2*v_0_1 - v_1_1 - 3*v_2_1 == 0  'load_def_1'
    l_0 - 4*s_0 <= 0  'capacity_0'
    l_1 - 5*s_1 <= 0  'capacity_1'


In [60]:
import os
import json
import time
import math
import traceback
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed, TimeoutError

import pandas as pd
import sys

# Try import local `cupy_utils` (one level up); provide NumPy fallback if missing
#sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), ".."))
try:
    import cupy_utils as cu
except Exception:
    import numpy as _np

    class _CuFallback:
        xp = _np

        @staticmethod
        def asarray(x):
            return _np.asarray(x)

        @staticmethod
        def asnumpy(x):
            return _np.asarray(x)

    cu = _CuFallback()


# ============================================================
# IMPORT QISKIT A LIVELLO MODULO
# ============================================================

In [61]:
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler
from qiskit_optimization.algorithms import MinimumEigenOptimizer, ADMMOptimizer
from qiskit_optimization.minimum_eigensolvers import QAOA
from qiskit_optimization.problems import QuadraticProgram

from qiskit_optimization.problems import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit_optimization.algorithms import ADMMOptimizer, ADMMParameters
from qiskit_optimization.algorithms.admm_optimizer import ADMMOptimizationResult


In [63]:
# ============================================================
# QUI DEVONO ESISTERE NEL TUO PROGETTO
# ============================================================
# - _build_qp(M, N, params)
# - _get_qubo_info(M, N, params)
# - _get_admm_params()
# - _decode(result, M, N)
# - _extract_residuals(result)
# - _compute_energy(decoded, M, N, params)
# - check_feasibility(result, M, N, params)
#
# Lascio i nomi invariati così puoi incollare questo file e
# importare o mantenere le tue funzioni originali.
# ============================================================

In [64]:
def _build_qp(M: int, N: int, params: dict) -> QuadraticProgram:
    """Costruisce il QuadraticProgram misto (binarie + continue) per (M, N)."""
    PI = params["P_idle"]
    PD = params["P_dynamic"]
    C = params["C_capacity"]
    U = params["u_cpu"]

    try:
        from docplex.mp.model import Model
        from qiskit_optimization.translators import from_docplex_mp

        mdl = Model(f"VM_Alloc_{M}x{N}")
        s = mdl.binary_var_list(M, name="s")
        v = [[mdl.binary_var(name=f"v_{j}_{i}") for i in range(M)] for j in range(N)]
        l = [mdl.continuous_var(lb=0, ub=C[i], name=f"l_{i}") for i in range(M)]

        mdl.minimize(
            mdl.sum(PI[i] * s[i] for i in range(M))
            + mdl.sum(PD[i] * l[i] for i in range(M))
        )

        for i in range(M):
            mdl.add_constraint(
                l[i] == mdl.sum(U[j] * v[j][i] for j in range(N)),
                ctname=f"load_def_{i}",
            )
        for i in range(M):
            mdl.add_constraint(l[i] <= C[i] * s[i], ctname=f"capacity_{i}")
        for j in range(N):
            mdl.add_constraint(
                mdl.sum(v[j][i] for i in range(M)) == 1,
                ctname=f"assign_{j}",
            )

        return from_docplex_mp(mdl)
    except ImportError:
        pass

    # Fallback manuale
    qp = QuadraticProgram(f"VM_Alloc_{M}x{N}")
    for i in range(M):
        qp.binary_var(name=f"s_{i}")
    for j in range(N):
        for i in range(M):
            qp.binary_var(name=f"v_{j}_{i}")
    for i in range(M):
        qp.continuous_var(lowerbound=0, upperbound=C[i], name=f"l_{i}")

    linear_obj = {}
    for i in range(M):
        linear_obj[f"s_{i}"] = PI[i]
        linear_obj[f"l_{i}"] = PD[i]
    qp.minimize(linear=linear_obj)

    for i in range(M):
        coeff = {f"l_{i}": 1}
        for j in range(N):
            coeff[f"v_{j}_{i}"] = -U[j]
        qp.linear_constraint(linear=coeff, sense="==", rhs=0, name=f"load_def_{i}")

    for i in range(M):
        coeff = {f"l_{i}": 1, f"s_{i}": -C[i]}
        qp.linear_constraint(linear=coeff, sense="<=", rhs=0, name=f"capacity_{i}")

    for j in range(N):
        coeff = {f"v_{j}_{i}": 1 for i in range(M)}
        qp.linear_constraint(linear=coeff, sense="==", rhs=1, name=f"assign_{j}")

    return qp

In [65]:
def _build_binary_only_qp(M: int, N: int, params: dict) -> QuadraticProgram:
    """Versione solo-binaria per analisi dimensionale QUBO."""
    PI = params["P_idle"]
    PD = params["P_dynamic"]
    C = params["C_capacity"]
    U = params["u_cpu"]

    qp = QuadraticProgram(f"VM_Alloc_bin_{M}x{N}")
    for i in range(M):
        qp.binary_var(name=f"s_{i}")
    for j in range(N):
        for i in range(M):
            qp.binary_var(name=f"v_{j}_{i}")

    linear_obj = {}
    for i in range(M):
        linear_obj[f"s_{i}"] = PI[i]
        for j in range(N):
            linear_obj[f"v_{j}_{i}"] = PD[i] * U[j]
    qp.minimize(linear=linear_obj)

    for i in range(M):
        coeff = {}
        for j in range(N):
            coeff[f"v_{j}_{i}"] = U[j]
        coeff[f"s_{i}"] = -C[i]
        qp.linear_constraint(linear=coeff, sense="<=", rhs=0, name=f"capacity_{i}")

    for j in range(N):
        coeff = {f"v_{j}_{i}": 1 for i in range(M)}
        qp.linear_constraint(linear=coeff, sense="==", rhs=1, name=f"assign_{j}")

    return qp

In [66]:
def _get_qubo_info(M: int, N: int, params: dict) -> dict:
    """Calcola dimensioni QUBO e sparsità."""
    n_binary = M + M * N
    qp_bin = _build_binary_only_qp(M, N, params)
    try:
        from performance_utils import convert_qubo_cached

        qubo = convert_qubo_cached(qp_bin)
        n_qubo = qubo.get_num_vars()
        n_slack = n_qubo - n_binary

        obj = qubo.objective
        Q_dict = obj.quadratic.to_dict()
        L_dict = obj.linear.to_dict()
        total_cells = n_qubo * n_qubo
        non_zero = len(Q_dict) + len(L_dict)
        sparsity = (non_zero / total_cells * 100) if total_cells > 0 else 0.0

        return {
            "n_qubo": n_qubo,
            "n_slack": n_slack,
            "n_qubits": n_qubo,
            "sparsity": round(sparsity, 2),
        }
    except Exception:
        return {
            "n_qubo": n_binary + M + N,
            "n_slack": M + N,
            "n_qubits": n_binary + M + N,
            "sparsity": 0.0,
        }


In [67]:
def _decode(result: ADMMOptimizationResult, M: int, N: int) -> dict:
    """Decodifica la soluzione x in s, v, l."""
    # Use cupy_utils to place arrays on GPU when available (best-effort)
    x_dev = cu.asarray(result.x)
    # Round on device
    s_dev = cu.xp.rint(x_dev[:M]).astype(int)
    v_flat_dev = cu.xp.rint(x_dev[M: M + N * M]).astype(int)
    # Bring back to numpy for compatibility with rest of code
    s = cu.asnumpy(s_dev).astype(int)
    v = cu.asnumpy(v_flat_dev).reshape(N, M)
    l = cu.asnumpy(x_dev[M + N * M: M + N * M + M])
    return {"s": s, "v": v, "l": l}

In [68]:
def _extract_residuals(result: ADMMOptimizationResult) -> tuple[list[float], int, bool]:
    """Estrae residui, numero iterazioni e flag convergenza."""
    state = getattr(result, "state", None)
    residuals: list[float] = []
    converged = False
    if state is not None:
        if hasattr(state, "residuals") and len(state.residuals) > 0:
            residuals = [float(r) for r in state.residuals]
        if hasattr(state, "converge"):
            converged = bool(state.converge)
    return residuals, len(residuals), converged

In [69]:
def _compute_energy(dec: dict, M: int, N: int, params: dict) -> tuple[float, int]:
    """Calcola energia totale e server accesi."""
    s, v = dec["s"], dec["v"]
    PI, PD, U = params["P_idle"], params["P_dynamic"], params["u_cpu"]
    total = 0.0
    servers_on = 0
    for i in range(M):
        if s[i] == 1:
            servers_on += 1
            total += PI[i]
            for j in range(N):
                total += PD[i] * U[j] * int(v[j][i])
    return total, servers_on

In [70]:
def check_feasibility(result: ADMMOptimizationResult, M: int, N: int, params: dict) -> bool:
    """
    Verifica manualmente che la soluzione soddisfi tutti i vincoli:
        1. Ogni VM assegnata a esattamente 1 server
        2. Capacità: carico ≤ C_i * s_i per ogni server
        3. Variabili binarie in {0, 1}
    """
    dec = _decode(result, M, N)
    s, v = dec["s"], dec["v"]
    C, U = params["C_capacity"], params["u_cpu"]

    # Assegnamento: ogni VM su esattamente 1 server
    for j in range(N):
        if v[j].sum() != 1:
            return False

    # Capacità
    for i in range(M):
        load = sum(U[j] * int(v[j][i]) for j in range(N))
        if load > C[i] * int(s[i]) + 1e-6:
            return False

    return True

In [71]:
def _get_admm_params() -> ADMMParameters:
    """Parametri ADMM condivisi (stessi del progetto principale)."""
    return ADMMParameters(
        rho_initial=10,
        factor_c=100000,
        beta=10000,
        maxiter=100,
        tol=1e-4,
        three_block=True,
    )

In [72]:
def run_configuration(
    M: int,
    N: int,
    params: dict,
    timeout_sec: int = 120,
    run_qaoa: bool = True,
    qaoa_max_qubits: int | None = 20,
    qaoa_reps: int = 1,
    qaoa_maxiter: int = 100,
) -> dict:
    """
    Esegue un singolo run (M, N): classico + QAOA.
    Restituisce un dict con metriche e diagnostica.
    """

    t_run0 = time.perf_counter()

    qp = _build_qp(M, N, params)
    qubo_info = _get_qubo_info(M, N, params)

    n_binary = M + M * N
    admm_params = _get_admm_params()

    # ── Classico ──
    numpy_solver = NumPyMinimumEigensolver()
    qubo_optimizer_cl = MinimumEigenOptimizer(numpy_solver)
    admm_cl = ADMMOptimizer(params=admm_params, qubo_optimizer=qubo_optimizer_cl)

    t0 = time.perf_counter()
    result_cl = admm_cl.solve(qp)
    cl_time = time.perf_counter() - t0

    dec_cl = _decode(result_cl, M, N)
    cl_residuals, cl_iter, cl_converged = _extract_residuals(result_cl)
    cl_energy, cl_servers_on = _compute_energy(dec_cl, M, N, params)
    cl_feasible = check_feasibility(result_cl, M, N, params)

    # ── QAOA ──
    qaoa_obj = None
    qaoa_iter = None
    qaoa_converged = False
    qaoa_time = None
    qaoa_residuals = None
    qaoa_energy = None
    qaoa_error = False
    qaoa_error_msg = ""
    qaoa_skipped = False
    qaoa_skip_reason = ""

    if not run_qaoa:
        qaoa_skipped = True
        qaoa_skip_reason = "run_qaoa=False"
    elif qaoa_max_qubits is not None and qubo_info["n_qubits"] > qaoa_max_qubits:
        qaoa_skipped = True
        qaoa_skip_reason = f"n_qubits>{qaoa_max_qubits}"
    else:
        try:
            sampler = StatevectorSampler()
            qaoa = QAOA(
                sampler=sampler,
                reps=qaoa_reps,
                optimizer=COBYLA(maxiter=qaoa_maxiter),
            )
            qubo_optimizer_qa = MinimumEigenOptimizer(qaoa)
            admm_qa = ADMMOptimizer(params=admm_params, qubo_optimizer=qubo_optimizer_qa)

            t0 = time.perf_counter()
            result_qa = admm_qa.solve(qp)
            qaoa_time = time.perf_counter() - t0

            dec_qa = _decode(result_qa, M, N)
            qaoa_residuals_list, qa_iter, qa_conv = _extract_residuals(result_qa)
            qa_energy, _ = _compute_energy(dec_qa, M, N, params)

            qaoa_obj = float(result_qa.fval)
            qaoa_iter = qa_iter
            qaoa_converged = qa_conv
            qaoa_residuals = qaoa_residuals_list
            qaoa_energy = qa_energy

        except Exception as e:
            qaoa_error = True
            qaoa_error_msg = str(e)

    # ── Metriche derivate ──
    obj_diff_pct = None
    iter_ratio = None
    time_ratio = None
    if qaoa_obj is not None and result_cl.fval != 0:
        obj_diff_pct = (qaoa_obj - result_cl.fval) / abs(result_cl.fval) * 100
    if qaoa_iter is not None and cl_iter > 0:
        iter_ratio = qaoa_iter / cl_iter
    if qaoa_time is not None and cl_time > 0:
        time_ratio = qaoa_time / cl_time

    total_wall = time.perf_counter() - t_run0

    return {
        "M": M,
        "N": N,
        "n_binary_vars_original": n_binary,
        "n_slack_vars": qubo_info["n_slack"],
        "n_qubo_vars": qubo_info["n_qubo"],
        "n_qubits": qubo_info["n_qubits"],
        "qubo_sparsity": qubo_info["sparsity"],
        "continuous_vars": M,
        "feasible": cl_feasible,

        "classical_obj": float(result_cl.fval),
        "classical_iter": cl_iter,
        "classical_converged": cl_converged,
        "classical_time_sec": round(cl_time, 4),
        "classical_residuals": cl_residuals,
        "classical_energy_total": cl_energy,
        "classical_servers_on": cl_servers_on,

        "qaoa_obj": qaoa_obj,
        "qaoa_iter": qaoa_iter,
        "qaoa_converged": qaoa_converged,
        "qaoa_time_sec": round(qaoa_time, 4) if qaoa_time is not None else None,
        "qaoa_residuals": qaoa_residuals,
        "qaoa_energy_total": qaoa_energy,
        "qaoa_error": qaoa_error,
        "qaoa_error_msg": qaoa_error_msg,
        "qaoa_skipped": qaoa_skipped,
        "qaoa_skip_reason": qaoa_skip_reason,

        "obj_diff_pct": round(obj_diff_pct, 4) if obj_diff_pct is not None else None,
        "iter_ratio": round(iter_ratio, 4) if iter_ratio is not None else None,
        "time_ratio": round(time_ratio, 4) if time_ratio is not None else None,

        "run_wall_time_sec": round(total_wall, 4),
    }


In [73]:
def _worker_run(job: tuple) -> dict:
    """
    Worker top-level, picklable.
    job = (M, N, params, timeout_sec, run_qaoa, qaoa_max_qubits, qaoa_reps, qaoa_maxiter)
    """
    M, N, params, timeout_sec, run_qaoa, qaoa_max_qubits, qaoa_reps, qaoa_maxiter = job
    started = time.perf_counter()

    try:
        result = run_configuration(
            M=M,
            N=N,
            params=params,
            timeout_sec=timeout_sec,
            run_qaoa=run_qaoa,
            qaoa_max_qubits=qaoa_max_qubits,
            qaoa_reps=qaoa_reps,
            qaoa_maxiter=qaoa_maxiter,
        )
        result["worker_status"] = "ok"
        result["worker_error"] = ""
    except Exception as e:
        result = {
            "M": M,
            "N": N,
            "worker_status": "error",
            "worker_error": str(e),
            "worker_traceback": traceback.format_exc(),

            "n_binary_vars_original": None,
            "n_slack_vars": None,
            "n_qubo_vars": None,
            "n_qubits": None,
            "qubo_sparsity": None,
            "continuous_vars": None,
            "feasible": None,

            "classical_obj": None,
            "classical_iter": None,
            "classical_converged": None,
            "classical_time_sec": None,
            "classical_residuals": None,
            "classical_energy_total": None,
            "classical_servers_on": None,

            "qaoa_obj": None,
            "qaoa_iter": None,
            "qaoa_converged": None,
            "qaoa_time_sec": None,
            "qaoa_residuals": None,
            "qaoa_energy_total": None,
            "qaoa_error": True,
            "qaoa_error_msg": str(e),
            "qaoa_skipped": None,
            "qaoa_skip_reason": "",

            "obj_diff_pct": None,
            "iter_ratio": None,
            "time_ratio": None,
        }

    result["worker_wall_time_sec"] = round(time.perf_counter() - started, 4)
    return result

In [74]:
def run_configurations_parallel(
    configurations: list[tuple[int, int]],
    params: dict,
    timeout_sec: int = 120,
    max_workers: int | None = None,
    run_qaoa: bool = True,
    qaoa_max_qubits: int | None = 20,
    qaoa_reps: int = 1,
    qaoa_maxiter: int = 100,
    global_timeout_sec: int | None = None,
    save_csv_path: str | None = None,
    save_json_path: str | None = None,
    verbose: bool = True,
) -> list[dict]:
    """
    Esegue in parallelo una lista di configurazioni [(M, N), ...].
    """

    if max_workers is None:
        cpu = os.cpu_count() or 2
        max_workers = max(1, cpu - 1)

    jobs = [
        (M, N, params, timeout_sec, run_qaoa, qaoa_max_qubits, qaoa_reps, qaoa_maxiter)
        for (M, N) in configurations
    ]

    if verbose:
        print(f"[PARALLEL] jobs={len(jobs)} max_workers={max_workers}")

    results = []
    t0 = time.perf_counter()

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        future_to_job = {executor.submit(_worker_run, job): job for job in jobs}

        try:
            iterator = as_completed(future_to_job, timeout=global_timeout_sec)
            for k, future in enumerate(iterator, start=1):
                job = future_to_job[future]
                M, N = job[0], job[1]

                try:
                    res = future.result()
                except Exception as e:
                    res = {
                        "M": M,
                        "N": N,
                        "worker_status": "future_error",
                        "worker_error": str(e),
                    }

                results.append(res)

                if verbose:
                    status = res.get("worker_status", "unknown")
                    ctime = res.get("classical_time_sec")
                    qtime = res.get("qaoa_time_sec")
                    print(
                        f"[DONE {k}/{len(jobs)}] "
                        f"(M={M}, N={N}) status={status} "
                        f"classical={ctime}s qaoa={qtime}s"
                    )

        except TimeoutError:
            if verbose:
                print("[PARALLEL] Global timeout reached while waiting for futures.")

            for future, job in future_to_job.items():
                if not future.done():
                    future.cancel()

    elapsed = time.perf_counter() - t0

    results.sort(key=lambda x: (x.get("M", math.inf), x.get("N", math.inf)))

    if verbose:
        print(f"[PARALLEL] total_wall_time_sec={elapsed:.4f}")

    if save_csv_path:
        df = pd.DataFrame(results)
        Path(save_csv_path).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(save_csv_path, index=False)

    if save_json_path:
        Path(save_json_path).parent.mkdir(parents=True, exist_ok=True)
        with open(save_json_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

    return results

In [75]:
def run_configurations_serial(
    configurations: list[tuple[int, int]],
    params: dict,
    timeout_sec: int = 120,
    run_qaoa: bool = True,
    qaoa_max_qubits: int | None = 20,
    qaoa_reps: int = 1,
    qaoa_maxiter: int = 100,
    save_csv_path: str | None = None,
    save_json_path: str | None = None,
    verbose: bool = True,
) -> list[dict]:
    """
    Versione seriale per benchmark e debug.
    """
    results = []
    t0 = time.perf_counter()

    for i, (M, N) in enumerate(configurations, start=1):
        if verbose:
            print(f"[SERIAL {i}/{len(configurations)}] M={M}, N={N}")

        res = _worker_run((M, N, params, timeout_sec, run_qaoa, qaoa_max_qubits, qaoa_reps, qaoa_maxiter))
        results.append(res)

    elapsed = time.perf_counter() - t0
    results.sort(key=lambda x: (x.get("M", math.inf), x.get("N", math.inf)))

    if verbose:
        print(f"[SERIAL] total_wall_time_sec={elapsed:.4f}")

    if save_csv_path:
        df = pd.DataFrame(results)
        Path(save_csv_path).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(save_csv_path, index=False)

    if save_json_path:
        Path(save_json_path).parent.mkdir(parents=True, exist_ok=True)
        with open(save_json_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

    return results

In [76]:
def summarize_results(results: list[dict]) -> dict:
    """
    Piccolo riassunto numerico per confrontare i batch.
    """
    if not results:
        return {}

    total = len(results)
    ok = sum(r.get("worker_status") == "ok" for r in results)
    err = sum(r.get("worker_status") != "ok" for r in results)
    qaoa_ok = sum((r.get("qaoa_obj") is not None) for r in results)
    qaoa_err = sum(bool(r.get("qaoa_error")) for r in results)
    qaoa_skipped = sum(bool(r.get("qaoa_skipped")) for r in results)

    classical_times = [r["classical_time_sec"] for r in results if r.get("classical_time_sec") is not None]
    qaoa_times = [r["qaoa_time_sec"] for r in results if r.get("qaoa_time_sec") is not None]
    wall_times = [r["worker_wall_time_sec"] for r in results if r.get("worker_wall_time_sec") is not None]

    return {
        "total_jobs": total,
        "worker_ok": ok,
        "worker_error": err,
        "qaoa_ok": qaoa_ok,
        "qaoa_error": qaoa_err,
        "qaoa_skipped": qaoa_skipped,
        "avg_classical_time_sec": round(sum(classical_times) / len(classical_times), 4) if classical_times else None,
        "avg_qaoa_time_sec": round(sum(qaoa_times) / len(qaoa_times), 4) if qaoa_times else None,
        "avg_worker_wall_time_sec": round(sum(wall_times) / len(wall_times), 4) if wall_times else None,
    }

In [77]:
if __name__ == "__main__":
    # ESEMPIO CONFIGURAZIONI
    configurations = [
        (2, 3),
        (3, 3),
        (3, 4),
        (4, 4),
        (4, 5),
        (5, 5),
    ]

    # SOSTITUISCI CON I TUOI PARAMETRI
    params = {
        # ...
    }

    # Benchmark seriale
    serial_results = run_configurations_serial(
        configurations=configurations,
        params=params,
        timeout_sec=120,
        run_qaoa=True,
        qaoa_max_qubits=20,
        qaoa_reps=1,
        qaoa_maxiter=80,
        save_csv_path="results/scaling_serial.csv",
        save_json_path="results/scaling_serial.json",
        verbose=True,
    )
    print("[SERIAL SUMMARY]", summarize_results(serial_results))

    # Benchmark parallelo
    parallel_results = run_configurations_parallel(
        configurations=configurations,
        params=params,
        timeout_sec=120,
        max_workers=max(1, (os.cpu_count() or 2) - 1),
        run_qaoa=True,
        qaoa_max_qubits=20,
        qaoa_reps=1,
        qaoa_maxiter=80,
        global_timeout_sec=None,
        save_csv_path="results/scaling_parallel.csv",
        save_json_path="results/scaling_parallel.json",
        verbose=True,
    )
    print("[PARALLEL SUMMARY]", summarize_results(parallel_results))


[SERIAL 1/6] M=2, N=3
[SERIAL 2/6] M=3, N=3
[SERIAL 3/6] M=3, N=4
[SERIAL 4/6] M=4, N=4
[SERIAL 5/6] M=4, N=5
[SERIAL 6/6] M=5, N=5
[SERIAL] total_wall_time_sec=0.0016
[SERIAL SUMMARY] {'total_jobs': 6, 'worker_ok': 0, 'worker_error': 6, 'qaoa_ok': 0, 'qaoa_error': 6, 'qaoa_skipped': 0, 'avg_classical_time_sec': None, 'avg_qaoa_time_sec': None, 'avg_worker_wall_time_sec': 0.0002}
[PARALLEL] jobs=6 max_workers=7
[DONE 1/6] (M=2, N=3) status=future_error classical=Nones qaoa=Nones
[DONE 2/6] (M=3, N=3) status=future_error classical=Nones qaoa=Nones
[DONE 3/6] (M=3, N=4) status=future_error classical=Nones qaoa=Nones
[DONE 4/6] (M=4, N=4) status=future_error classical=Nones qaoa=Nones
[DONE 5/6] (M=4, N=5) status=future_error classical=Nones qaoa=Nones
[DONE 6/6] (M=5, N=5) status=future_error classical=Nones qaoa=Nones
[PARALLEL] total_wall_time_sec=1.1180
[PARALLEL SUMMARY] {'total_jobs': 6, 'worker_ok': 0, 'worker_error': 6, 'qaoa_ok': 0, 'qaoa_error': 0, 'qaoa_skipped': 0, 'avg_classi

In [78]:
"""
config_generator.py
===================
Genera tutte le combinazioni (M, N) per lo scaling study e, per ciascuna,
un set di parametri realistici deterministici (seed = M*100 + N).
"""

'\nconfig_generator.py\n===================\nGenera tutte le combinazioni (M, N) per lo scaling study e, per ciascuna,\nun set di parametri realistici deterministici (seed = M*100 + N).\n'

In [79]:
from __future__ import annotations

import random

MAX_M = 4   # numero massimo di server
MAX_N = 5   # numero massimo di VM

In [80]:
def generate_all_configs(
    max_M: int = MAX_M,
    max_N: int = MAX_N,
) -> list[dict]:
    """
    Restituisce una lista di dict, uno per ogni coppia (M, N).

    Ogni dict contiene:
        M, N, P_idle, P_dynamic, C_capacity, u_cpu
    """
    configs: list[dict] = []

    for M in range(1, max_M + 1):
        for N in range(1, max_N + 1):
            rng = random.Random(M * 100 + N)

            P_idle = [rng.randint(80, 150) for _ in range(M)]
            P_dynamic = [rng.randint(40, 80) for _ in range(M)]
            C_capacity = [rng.randint(4, 10) for _ in range(M)]
            c_min = min(C_capacity)

            # Garantisci che ogni VM abbia carico ≤ c_min − 1 (almeno 1)
            u_max = max(c_min - 1, 1)
            u_cpu = [rng.randint(1, u_max) for _ in range(N)]

            configs.append({
                "M": M,
                "N": N,
                "P_idle": P_idle,
                "P_dynamic": P_dynamic,
                "C_capacity": C_capacity,
                "u_cpu": u_cpu,
            })

    return configs

In [81]:
def print_config_summary(configs: list[dict]) -> None:
    """Stampa un riepilogo delle configurazioni generate."""
    print(f"\n{'=' * 60}")
    print(f"  PIANO DI ESECUZIONE — Scaling Study")
    print(f"{'=' * 60}")
    print(f"  Configurazioni totali: {len(configs)}")
    print(f"  Range M (server):      1 → {max(c['M'] for c in configs)}")
    print(f"  Range N (VM):          1 → {max(c['N'] for c in configs)}")
    print(f"\n  {'M':>3} {'N':>3}  {'Bin vars':>9}  {'Cont':>5}  {'Tot orig':>9}  {'P_idle':>18}  {'u_cpu':>18}")
    print(f"  {'-' * 80}")
    for c in configs:
        n_bin = c["M"] + c["M"] * c["N"]
        n_cont = c["M"]
        print(
            f"  {c['M']:>3} {c['N']:>3}  {n_bin:>9}  {n_cont:>5}  {n_bin + n_cont:>9}"
            f"  {str(c['P_idle']):>18}  {str(c['u_cpu']):>18}"
        )
    print(f"{'=' * 60}\n")

In [82]:
if __name__ == "__main__":
    configs = generate_all_configs()
    print_config_summary(configs)


  PIANO DI ESECUZIONE — Scaling Study
  Configurazioni totali: 20
  Range M (server):      1 → 4
  Range N (VM):          1 → 5

    M   N   Bin vars   Cont   Tot orig              P_idle               u_cpu
  --------------------------------------------------------------------------------
    1   1          2      1          3               [104]                 [4]
    1   2          3      1          4                [98]              [2, 5]
    1   3          4      1          5               [138]           [2, 2, 7]
    1   4          5      1          6                [82]        [4, 7, 6, 1]
    1   5          6      1          7               [124]     [2, 1, 5, 9, 6]
    2   1          4      2          6           [88, 121]                 [3]
    2   2          6      2          8          [129, 140]              [4, 1]
    2   3          8      2         10            [91, 85]           [3, 2, 3]
    2   4         10      2         12           [130, 90]        [4, 3, 1, 

In [83]:
"""
scaling_runner.py
=================
Entry point dello scaling study. Itera tutte le configurazioni (M, N),
esegue il solver ADMM classico + QAOA per ciascuna e salva i risultati
in scaling_results.json.
"""

'\nscaling_runner.py\n=================\nEntry point dello scaling study. Itera tutte le configurazioni (M, N),\nesegue il solver ADMM classico + QAOA per ciascuna e salva i risultati\nin scaling_results.json.\n'

In [87]:
from __future__ import annotations

import json
import os
import sys
import time
from datetime import datetime

# Aggiungi directory corrente e QAOA al path
#HERE = os.path.dirname(os.path.abspath(__file__))
#sys.path.insert(0, HERE)
#sys.path.insert(0, os.path.join(HERE, ".."))

#from config_generator import generate_all_configs, print_config_summary
#from single_run import run_configuration

In [88]:
def _progress_bar(current: int, total: int, width: int = 30, label: str = "") -> str:
    pct = current / total
    filled = int(width * pct)
    bar = "█" * filled + "░" * (width - filled)
    return f"  [{bar}] {current}/{total} ({pct * 100:.0f}%) — {label}"


In [89]:
def main() -> None:
    print("╔" + "═" * 68 + "╗")
    print("║         SCALING STUDY — QAOA VM Allocation                       ║")
    print("╚" + "═" * 68 + "╝")

    configs = generate_all_configs()
    print_config_summary(configs)

    total = len(configs)
    results: list[dict] = []
    n_qaoa_ok = 0

    t_start = time.perf_counter()

    for idx, cfg in enumerate(configs, start=1):
        M, N = cfg["M"], cfg["N"]
        label = f"Running M={M}, N={N} ..."
        print(_progress_bar(idx, total, label=label), flush=True)

        try:
            r = run_configuration(M, N, cfg, timeout_sec=120)
        except Exception as e:
            # Fallback: salva un entry con errore completo
            r = {
                "M": M, "N": N,
                "n_binary_vars_original": M + M * N,
                "n_slack_vars": 0, "n_qubo_vars": 0, "n_qubits": 0,
                "qubo_sparsity": 0.0, "continuous_vars": M, "feasible": False,
                "classical_obj": 0.0, "classical_iter": 0,
                "classical_converged": False, "classical_time_sec": 0.0,
                "classical_residuals": [], "classical_energy_total": 0.0,
                "classical_servers_on": 0,
                "qaoa_obj": None, "qaoa_iter": None, "qaoa_converged": False,
                "qaoa_time_sec": None, "qaoa_residuals": None,
                "qaoa_energy_total": None, "qaoa_error": True,
                "qaoa_error_msg": str(e),
                "obj_diff_pct": None, "iter_ratio": None, "time_ratio": None,
            }
            print(f"    ⚠️  Config M={M}, N={N} fallita completamente: {e}")

        if not r.get("qaoa_error", True):
            n_qaoa_ok += 1
        results.append(r)

    elapsed = time.perf_counter() - t_start

    n_classical_ok = sum(1 for r in results if r["classical_iter"] > 0)

    # ── Salvataggio ──
    output = {
        "metadata": {
            "generated_at": datetime.now().isoformat(),
            "total_configs": total,
            "successful_classical": n_classical_ok,
            "successful_qaoa": n_qaoa_ok,
            "qiskit_optimization_version": "0.7.0",
            "max_M": max(c["M"] for c in configs),
            "max_N": max(c["N"] for c in configs),
            "total_time_sec": round(elapsed, 2),
        },
        "results": results,
    }

    out_path = os.path.join(HERE, "scaling_results.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print(f"\n{'=' * 60}")
    print(f"  RIEPILOGO SCALING STUDY")
    print(f"{'=' * 60}")
    print(f"  Tempo totale:            {elapsed:.1f} s")
    print(f"  Config classico OK:      {n_classical_ok}/{total}")
    print(f"  Config QAOA OK:          {n_qaoa_ok}/{total}")

    # Config con più qubits
    max_qubits_r = max(results, key=lambda r: r["n_qubits"])
    print(f"\n  🔬 QUBO più grande:      M={max_qubits_r['M']}, N={max_qubits_r['N']} "
          f"→ {max_qubits_r['n_qubits']} qubit")

    # Config con maggior deviazione QAOA
    valid_diff = [r for r in results if r["obj_diff_pct"] is not None]
    if valid_diff:
        worst_diff = max(valid_diff, key=lambda r: abs(r["obj_diff_pct"]))
        print(f"  📊 Max diff QAOA:        M={worst_diff['M']}, N={worst_diff['N']} "
              f"→ {worst_diff['obj_diff_pct']:.2f}%")

    # Config più lenta QAOA
    valid_time = [r for r in results if r["qaoa_time_sec"] is not None]
    if valid_time:
        slowest = max(valid_time, key=lambda r: r["qaoa_time_sec"])
        print(f"  ⏱️  QAOA più lento:      M={slowest['M']}, N={slowest['N']} "
              f"→ {slowest['qaoa_time_sec']:.2f} s")

    # Feasibility
    n_feasible = sum(1 for r in results if r["feasible"])
    print(f"  ✅ Soluzioni feasible:   {n_feasible}/{total}")

    print(f"\n  [JSON] Risultati salvati in: {out_path}")
    print(f"{'=' * 60}\n")


In [ ]:
if __name__ == "__main__":
    main()

╔════════════════════════════════════════════════════════════════════╗
║         SCALING STUDY — QAOA VM Allocation                       ║
╚════════════════════════════════════════════════════════════════════╝

  PIANO DI ESECUZIONE — Scaling Study
  Configurazioni totali: 20
  Range M (server):      1 → 4
  Range N (VM):          1 → 5

    M   N   Bin vars   Cont   Tot orig              P_idle               u_cpu
  --------------------------------------------------------------------------------
    1   1          2      1          3               [104]                 [4]
    1   2          3      1          4                [98]              [2, 5]
    1   3          4      1          5               [138]           [2, 2, 7]
    1   4          5      1          6                [82]        [4, 7, 6, 1]
    1   5          6      1          7               [124]     [2, 1, 5, 9, 6]
    2   1          4      2          6           [88, 121]                 [3]
    2   2          6   